# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
BRANCH = "main"
!cd {PROJECT_ROOT} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
!cd {PROJECT_ROOT} && git branch --show-current && git log -1 --oneline


# If this notebook itself was modified on Drive, open a *temporary* notebook and run:
#
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
# BRANCH = "main"
# !cd {PROJECT_ROOT} && git reset --hard && git fetch --all && git checkout {BRANCH} && git pull origin {BRANCH}
# !cd {PROJECT_ROOT} && git branch --show-current && git log -1 --oneline

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps    = 60000,
    batch_size   = 32,
    seed         = 42,
    checkpoint   = 200,
    early_stop   = 50,

    # ── regularisation ───────────────────────────────────────────────────
    dropout      = 0.3,
    dropout_char = 0.15,

    # ── paper-aligned recipe: Adam + lambda warmup ───────────────────────
    optimizer_name = "adam",
    scheduler_name = "lambda",
    loss_name      = "qa_nll",
    ema_decay      = 0.9999,
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

---
## Section 5 — Stage 3: Experimental Investigation

Eval-time mechanistic analysis adapted from Meng et al. (NeurIPS 2022) "Locating and Editing Factual Associations in GPT" to probe QANet's internal mechanisms.

- **H1**: Mechanistic divergence between architecturally identical Conv_0 and Conv_1 in the Model Encoder
- **H2**: Directional asymmetry in CQ Attention — C→Q information injection vs Q→C redistribution
- **H3**: Necessity of Pointer asymmetric wiring — testing functional specialisation across M1/M2/M3

In [ ]:
## Stage 3 — Setup: Load trained model for analysis

!pip install matplotlib seaborn -q

import argparse, json, os, random
import numpy as np
import torch
import torch.nn.functional as F
from Data import SQuADDataset, load_word_char_mats, load_train_dev_eval, make_loader
from Models import QANet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_PATH = "_model/model_best.pt"
DEV_NPZ = "_data/dev.npz"
DEV_EVAL_JSON = "_data/dev_eval.json"

# Load checkpoint and rebuild model
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
config = ckpt["config"]
model_args = argparse.Namespace(**config)

word_mat, char_mat = load_word_char_mats(model_args)
model = QANet(word_mat, char_mat, model_args).to(DEVICE)
state_key = "model_state" if "model_state" in ckpt else "model_state_dict"
model.load_state_dict(ckpt[state_key])
model.eval()

dataset = SQuADDataset(DEV_NPZ)
import ujson
with open(DEV_EVAL_JSON) as f:
    eval_file = ujson.load(f)

num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Device:           {DEVICE}")
print(f"Checkpoint:       {CKPT_PATH}  (key: '{state_key}')")
print(f"Dev set:          {len(dataset)} samples")
print(f"Model params:     {num_params:,} total, {num_trainable:,} trainable")
print(f"Checkpoint F1:    {ckpt.get('best_f1', '?')}")
print(f"Checkpoint EM:    {ckpt.get('best_em', '?')}")
print(f"Config summary:   d_model={config.get('d_model','?')}, "
      f"num_head={config.get('num_head','?')}, "
      f"emb_enc_blocks=1×{config.get('emb_enc_conv_num', config.get('conv_num','?'))} conv, "
      f"model_enc_blocks=7×2 conv×3 passes")

### H1: Mechanistic Divergence Between Architecturally Identical Conv_0 and Conv_1

**Research question**: Conv_0 and Conv_1 share identical architecture (depthwise separable, k=5) yet exhibit an order-of-magnitude importance gap (ΔF1 ≈ 30×). What mechanistic differences explain this?

**Motivation**: Global ablation reveals Conv_1 is the most important component (−32.54 F1) while Conv_0 is the least (−1.09 F1). Architecture alone cannot explain this; the divergence must arise from pipeline position and learned function.

**Three methods**:
- **Method 1**: Eval-time global ablation — skip all Conv_0 vs Conv_1 vs Self-Attn vs FFN, measure F1/EM
- **Method 2**: ROME-style causal tracing — corrupt input, restore individual sub-layers, measure AIE
- **Method 3**: Per-block ablation — skip component in specific (pass, block), measure F1 drop

In [ ]:
from experiments.tracer import (
    qanet_forward, compute_span_prob, compute_start_prob, compute_end_prob,
    build_model_enc_specs, RestoreSpec,
)
from tqdm.auto import tqdm

NUM_SAMPLES_H1 = 200       # reduce if slow
NOISE_STD_SCALE = 3.0
NOISE_REPEATS = 3
MIN_CLEAN_PROB = 0.01
SEED = 42

random.seed(SEED); torch.manual_seed(SEED)
loader_h1 = make_loader(dataset, batch_size=1, shuffle=True)
specs_h1 = build_model_enc_specs()
spec_keys_h1 = [f"p{s.pass_idx}_b{s.block_idx}_{s.component}" for s in specs_h1]

h1_ie = {k: [] for k in spec_keys_h1}
h1_ie_p1 = {k: [] for k in spec_keys_h1}
h1_ie_p2 = {k: [] for k in spec_keys_h1}
h1_te_list = []
samples_used = 0

pbar = tqdm(total=NUM_SAMPLES_H1, desc="H1 Causal Tracing")
with torch.no_grad():
    for batch_idx, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_h1):
        if samples_used >= NUM_SAMPLES_H1:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        # Clean run
        p1_c, p2_c, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        rep_ie = {k: [] for k in spec_keys_h1}
        rep_ie_p1 = {k: [] for k in spec_keys_h1}
        rep_ie_p2 = {k: [] for k in spec_keys_h1}
        rep_te = []

        for rep in range(NOISE_REPEATS):
            ns = SEED + batch_idx * 100 + rep
            p1_x, p2_x, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                              corrupt_target="context", noise_std_scale=NOISE_STD_SCALE, noise_seed=ns)
            prob_corrupt = compute_span_prob(p1_x, p2_x, y1, y2)
            prob_p1_x = compute_start_prob(p1_x, y1)
            prob_p2_x = compute_end_prob(p2_x, y2)
            te = (prob_clean - prob_corrupt).item()
            rep_te.append(te)
            if abs(te) < 1e-6:
                continue

            for spec, key in zip(specs_h1, spec_keys_h1):
                p1_r, p2_r, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                  corrupt_target="context", noise_std_scale=NOISE_STD_SCALE, noise_seed=ns,
                                                  clean_acts=clean_acts, restore_spec=spec)
                prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                rep_ie[key].append((prob_r - prob_corrupt).item())
                rep_ie_p1[key].append((compute_start_prob(p1_r, y1) - prob_p1_x).item())
                rep_ie_p2[key].append((compute_end_prob(p2_r, y2) - prob_p2_x).item())

        avg_te = np.mean(rep_te) if rep_te else 0.0
        h1_te_list.append(avg_te)
        for key in spec_keys_h1:
            if rep_ie[key]:
                h1_ie[key].append(np.mean(rep_ie[key]))
                h1_ie_p1[key].append(np.mean(rep_ie_p1[key]))
                h1_ie_p2[key].append(np.mean(rep_ie_p2[key]))
        samples_used += 1
        pbar.update(1)
        pbar.set_postfix(TE=f"{np.mean(h1_te_list):.4f}", skipped=batch_idx+1-samples_used)
pbar.close()

# Aggregate
h1_results = {}
for key in spec_keys_h1:
    vs = np.array(h1_ie[key])
    n = len(vs)
    h1_results[key] = {
        "aie_span": float(vs.mean()) if n else 0.0,
        "aie_p1": float(np.mean(h1_ie_p1[key])) if n else 0.0,
        "aie_p2": float(np.mean(h1_ie_p2[key])) if n else 0.0,
        "ci95": float(vs.std() * 1.96 / np.sqrt(n)) if n > 1 else 0.0,
        "n": n,
    }

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_results.json", "w") as f:
    json.dump({"results": h1_results, "meta": {"num_samples": samples_used, "avg_te": float(np.mean(h1_te_list))}}, f, indent=2)

print(f"\n{'='*70}")
print(f"{'='*70}")
print(f"Samples used: {samples_used} / {NUM_SAMPLES_H1}  |  Noise repeats: {NOISE_REPEATS}  |  σ scale: {NOISE_STD_SCALE}")
print(f"Average Total Effect (TE): {np.mean(h1_te_list):.4f} ± {np.std(h1_te_list)*1.96/np.sqrt(len(h1_te_list)):.4f}")
print(f"TE range: [{np.min(h1_te_list):.4f}, {np.max(h1_te_list):.4f}]  median: {np.median(h1_te_list):.4f}")

components = ["conv_0", "conv_1", "self_attn", "ffn"]
comp_labels = {"conv_0": "Conv-0", "conv_1": "Conv-1", "self_attn": "Self-Attn", "ffn": "FFN"}

print(f"\n--- Aggregate AIE by Component Type ---")
print(f"{'Component':<14s}  {'Mean AIE':>10s}  {'Std':>8s}  {'Sum AIE':>10s}  {'Count':>6s}")
print("-" * 55)
for c in components:
    vals = [h1_results[k]["aie_span"] for k in h1_results if k.endswith(c)]
    print(f"{comp_labels[c]:<14s}  {np.mean(vals):10.4f}  {np.std(vals):8.4f}  {np.sum(vals):10.4f}  {len(vals):6d}")

conv_sum = sum(h1_results[k]["aie_span"] for k in h1_results if "conv" in k)
attn_sum = sum(h1_results[k]["aie_span"] for k in h1_results if k.endswith("self_attn"))
ffn_sum = sum(h1_results[k]["aie_span"] for k in h1_results if k.endswith("ffn"))
total_sum = conv_sum + attn_sum + ffn_sum
print(f"\nTotal AIE share:  Conv(0+1) = {conv_sum:.4f} ({conv_sum/max(total_sum,1e-8):.1%}),  "
      f"Attn = {attn_sum:.4f} ({attn_sum/max(total_sum,1e-8):.1%}),  FFN = {ffn_sum:.4f} ({ffn_sum/max(total_sum,1e-8):.1%})")

print(f"\n--- Aggregate AIE by Pass ---")
for p in range(3):
    p_vals = [h1_results[k]["aie_span"] for k in h1_results if k.startswith(f"p{p}_")]
    print(f"  Pass {p} → M{p+1}: mean = {np.mean(p_vals):.4f}, sum = {np.sum(p_vals):.4f}, max = {np.max(p_vals):.4f}")

print(f"\n--- Aggregate AIE by Block Depth ---")
for b in range(7):
    b_vals = [h1_results[k]["aie_span"] for k in h1_results if f"_b{b}_" in k]
    print(f"  Block {b}: mean = {np.mean(b_vals):.4f}, sum = {np.sum(b_vals):.4f}")

print(f"\n--- Top 10 Sub-layers by AIE (span) ---")
print(f"{'Rank':<5s} {'Sub-layer':<25s}  {'AIE(span)':>10s}  {'±95%CI':>8s}  {'AIE(p1)':>8s}  {'AIE(p2)':>8s}  {'p1−p2':>8s}  {'N':>4s}")
print("-" * 80)
ranked = sorted(h1_results.items(), key=lambda x: -x[1]["aie_span"])
for rank, (k, r) in enumerate(ranked[:10], 1):
    diff_p = r["aie_p1"] - r["aie_p2"]
    print(f"{rank:<5d} {k:<25s}  {r['aie_span']:10.4f}  {r['ci95']:8.4f}  {r['aie_p1']:8.4f}  {r['aie_p2']:8.4f}  {diff_p:+8.4f}  {r['n']:4d}")

print(f"\n--- Bottom 5 Sub-layers (least causal importance) ---")
for k, r in ranked[-5:]:
    print(f"  {k:<25s}  AIE = {r['aie_span']:.4f} ± {r['ci95']:.4f}")

In [ ]:
## H1 — Norm Diagnostic: Are we measuring importance or output magnitude?
##
## If Conv_1 has much larger output norm than Conv_0, then zero-out ablation
## and causal tracing may simply reflect norm differences, not causal importance.

from experiments.tracer import qanet_forward

N_NORM_SAMPLES = 200
loader_norm = make_loader(dataset, batch_size=1, shuffle=True)

comp_names = ["conv_0", "conv_1", "self_attn", "ffn"]
norm_acc = {}  # key: "p{pass}_b{block}_{comp}" -> list of norms

random.seed(42); torch.manual_seed(42)
samples_norm = 0

with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_norm, total=N_NORM_SAMPLES, desc="Norm diagnostic"):
        if samples_norm >= N_NORM_SAMPLES:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        model_enc_acts = clean_acts.get("model_enc", {})
        for pass_idx in range(3):
            pass_acts = model_enc_acts.get(f"pass_{pass_idx}", {})
            for block_idx in range(7):
                blk_acts = pass_acts.get(f"block_{block_idx}", {})
                for comp in comp_names:
                    act = blk_acts.get(comp)
                    if act is None:
                        continue
                    key = f"p{pass_idx}_b{block_idx}_{comp}"
                    norm_val = act.float().norm().item()
                    norm_acc.setdefault(key, []).append(norm_val)
        samples_norm += 1

print(f"\n{'='*70}")
print("NORM DIAGNOSTIC — Sub-layer Output L2 Norms")
print(f"{'='*70}")
print(f"Samples: {samples_norm}")

# Aggregate by component type
comp_norms = {c: [] for c in comp_names}
for key, vals in norm_acc.items():
    for c in comp_names:
        if key.endswith(c):
            comp_norms[c].append(np.mean(vals))

print(f"{'Component':<14s}  {'Mean norm':>10s}  {'Std':>10s}  {'Ratio vs Conv_0':>16s}")
print("-" * 55)
conv0_mean = np.mean(comp_norms["conv_0"]) if comp_norms["conv_0"] else 1e-8
for c in comp_names:
    vals = comp_norms[c]
    m = np.mean(vals)
    s = np.std(vals)
    ratio = m / conv0_mean
    print(f"{c:<14s}  {m:10.2f}  {s:10.2f}  {ratio:15.2f}x")

# Aggregate by pass
print(f"\n{'Pass':<10s}  {'Conv_0':>10s}  {'Conv_1':>10s}  {'Attn':>10s}  {'FFN':>10s}")
print("-" * 55)
for p in range(3):
    row = []
    for c in comp_names:
        vals = [np.mean(norm_acc[k]) for k in norm_acc if k.startswith(f"p{p}_") and k.endswith(c)]
        row.append(np.mean(vals) if vals else 0)
    print(f"Pass {p:<5d}  {row[0]:10.2f}  {row[1]:10.2f}  {row[2]:10.2f}  {row[3]:10.2f}")

# Aggregate by block depth
print(f"\n{'Block':<10s}  {'Conv_0':>10s}  {'Conv_1':>10s}  {'Attn':>10s}  {'FFN':>10s}")
print("-" * 55)
for b in range(7):
    row = []
    for c in comp_names:
        vals = [np.mean(norm_acc[k]) for k in norm_acc if f"_b{b}_" in k and k.endswith(c)]
        row.append(np.mean(vals) if vals else 0)
    print(f"Block {b:<5d}  {row[0]:10.2f}  {row[1]:10.2f}  {row[2]:10.2f}  {row[3]:10.2f}")

# The critical test: correlation between norm and AIE
print(f"\n--- Critical Test: Norm vs AIE Correlation ---")
if os.path.exists("experiments/results/H1/h1_results.json"):
    with open("experiments/results/H1/h1_results.json") as f:
        h1_res_check = json.load(f)["results"]

    norm_vals_corr, aie_vals_corr, keys_corr = [], [], []
    for key in h1_res_check:
        if key in norm_acc:
            norm_vals_corr.append(np.mean(norm_acc[key]))
            aie_vals_corr.append(h1_res_check[key]["aie_span"])
            keys_corr.append(key)

    norm_vals_corr = np.array(norm_vals_corr)
    aie_vals_corr = np.array(aie_vals_corr)

    if len(norm_vals_corr) > 2:
        r_norm_aie = np.corrcoef(norm_vals_corr, aie_vals_corr)[0, 1]
        print(f"Pearson r(norm, AIE) = {r_norm_aie:.3f}  (n={len(norm_vals_corr)} sub-layers)")
        if abs(r_norm_aie) > 0.7:
        elif abs(r_norm_aie) > 0.4:
        else:

        # ── Norm-Normalized Analysis ──
        aie_per_norm = aie_vals_corr / np.maximum(norm_vals_corr, 1e-8)
        r_after = np.corrcoef(norm_vals_corr, aie_per_norm)[0, 1]

        print(f"\n{'='*70}")
        print("NORM-NORMALIZED AIE (AIE / output_norm)")
        print(f"{'='*70}")
        print(f"After normalization: r(norm, AIE/norm) = {r_after:.3f}")
        if abs(r_after) < 0.3:
        else:

        # Aggregate normalized AIE by component type
        comp_naie = {c: [] for c in comp_names}
        for i, key in enumerate(keys_corr):
            for c in comp_names:
                if key.endswith(c):
                    comp_naie[c].append(aie_per_norm[i])

        print(f"{'Component':<14s}  {'Raw AIE':>10s}  {'Mean norm':>10s}  {'AIE/norm':>12s}  {'Rank change'}")
        print("-" * 70)
        raw_ranking = sorted(comp_names, key=lambda c: -np.mean([aie_vals_corr[i] for i, k in enumerate(keys_corr) if k.endswith(c)]))
        norm_ranking = sorted(comp_names, key=lambda c: -np.mean(comp_naie[c]) if comp_naie[c] else 0)
        for c in comp_names:
            raw = np.mean([aie_vals_corr[i] for i, k in enumerate(keys_corr) if k.endswith(c)])
            normed = np.mean(comp_naie[c]) if comp_naie[c] else 0
            mn = np.mean([norm_vals_corr[i] for i, k in enumerate(keys_corr) if k.endswith(c)])
            old_rank = raw_ranking.index(c) + 1
            new_rank = norm_ranking.index(c) + 1
            change = "—" if old_rank == new_rank else f"{old_rank}→{new_rank}"
            print(f"{c:<14s}  {raw:10.4f}  {mn:10.1f}  {normed:12.6f}  {change}")

        print(f"\nRaw ranking:       {' > '.join(raw_ranking)}")
        print(f"Normalized ranking: {' > '.join(norm_ranking)}")
        if raw_ranking == norm_ranking:
        else:

        # Top 10 by normalized AIE
        sorted_idx = np.argsort(-aie_per_norm)
        print(f"\n--- Top 10 Sub-layers by Norm-Normalized AIE ---")
        print(f"{'Rank':<5s} {'Sub-layer':<25s}  {'AIE':>8s}  {'Norm':>8s}  {'AIE/norm':>10s}")
        print("-" * 62)
        for rank, idx in enumerate(sorted_idx[:10], 1):
            print(f"{rank:<5d} {keys_corr[idx]:<25s}  {aie_vals_corr[idx]:8.4f}  {norm_vals_corr[idx]:8.1f}  {aie_per_norm[idx]:10.6f}")

        # Save norm data for later use
        norm_summary = {key: float(np.mean(norm_acc[key])) for key in norm_acc}
        with open("experiments/results/H1/h1_norms.json", "w") as f:
            json.dump({"norms": norm_summary, "r_norm_aie": float(r_norm_aie), "r_after_normalization": float(r_after)}, f, indent=2)
else:
    print("(H1 results not yet saved — run Method 2 first)")

In [ ]:
## H1 — Intervention Spectrum: Zero / Noise / Mean Replacement
##
## Four conditions to separate information content from magnitude:
##   original  — no intervention (baseline)
##   mean      — replace with dataset-average activation (in-distribution, removes sample-specific info)
##   noise     — replace with norm-matched random noise (out-of-distribution, removes all info)
##   zero      — replace with zeros (removes info + norm)
##
## Expected ordering if SAMPLE-SPECIFIC INFORMATION drives importance:
##   damage: zero ≈ noise > mean >> original
## Expected ordering if only NORM drives importance:
##   damage: zero >> noise ≈ mean (noise/mean preserve norm, so less shift)
##
## LIMITATIONS:
##   1. Mean computed from 500 samples; approximates population mean.
##   2. Sequence lengths vary; mean activation is truncated to match each sample.
##   3. Mean replacement is in-distribution on average but each specific sample sees a
##      mis-matched activation — this still introduces some distributional shift,
##      so the "mean damage" is an UPPER BOUND on the true fixed-contribution damage.
##   4. All interventions are global (all 21 blocks simultaneously), so results reflect
##      collective rather than per-block contribution.

import importlib, experiments.tracer, random
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, SkipSpec
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate
from tqdm.auto import tqdm

print("=" * 70)
print("=" * 70)

# ── Step 1: Collect mean activations across 500 samples ──
N_MEAN = 500
loader_mean = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)
mean_sums = {}
mean_counts = {}
comp_list = ["conv_0", "conv_1", "self_attn", "ffn"]

samples_mean = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_mean, total=N_MEAN, desc="Collecting means"):
        if samples_mean >= N_MEAN:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        _, _, acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        model_enc = acts.get("model_enc", {})
        for pi in range(3):
            for bi in range(7):
                blk = model_enc.get(f"pass_{pi}", {}).get(f"block_{bi}", {})
                for c in comp_list:
                    act = blk.get(c)
                    if act is None:
                        continue
                    key = (pi, bi, c)
                    if key not in mean_sums:
                        mean_sums[key] = torch.zeros_like(act[0])
                        mean_counts[key] = 0
                    L = min(mean_sums[key].shape[-1], act.shape[-1])
                    mean_sums[key][..., :L] += act[0, ..., :L].float()
                    mean_counts[key] += 1
        samples_mean += 1

mean_acts = {}
for key, s in mean_sums.items():
    mean_acts[key] = (s / mean_counts[key]).to(DEVICE)

print(f"Collected means from {samples_mean} samples for {len(mean_acts)} (pass, block, comp) positions.\n")

# ── Step 2: Subsampled baseline ──
MAX_BATCHES_NR = 64
loader_base_nr = make_loader(dataset, batch_size=32, shuffle=False)
answers_base_nr = {}
with torch.no_grad():
    for bi, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(tqdm(loader_base_nr, desc="  baseline", total=MAX_BATCHES_NR)):
        if bi >= MAX_BATCHES_NR:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=None)
        p1_log = F.log_softmax(p1, dim=1)
        p2_log = F.log_softmax(p2, dim=1)
        outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
        tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
        outer = outer.masked_fill(~tri, float('-inf'))
        flat = outer.view(outer.size(0), -1)
        idx = torch.argmax(flat, dim=1)
        L = p1.size(1)
        ymin = idx // L; ymax = idx % L
        ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
        answers_base_nr.update(ans)
nr_base = squad_evaluate(eval_file, answers_base_nr)
nr_base_f1 = nr_base["f1"]
nr_base_em = nr_base["exact_match"]
print(f"Subsampled baseline ({len(answers_base_nr)} samples): F1={nr_base_f1:.2f}, EM={nr_base_em:.2f}\n")

# ── Step 3: Run all intervention conditions ──
INTERV_CONFIGS = {}
for comp, skip_key in [("conv_1", "conv_1"), ("conv_0", "conv_0"), ("attn", "self_attn"), ("ffn", "ffn")]:
    INTERV_CONFIGS[f"zero_{comp}"]  = SkipSpec(global_skip={skip_key}, mode="zero")
    INTERV_CONFIGS[f"noise_{comp}"] = SkipSpec(global_skip={skip_key}, mode="noise")
    INTERV_CONFIGS[f"mean_{comp}"]  = SkipSpec(global_skip={skip_key}, mode="mean", mean_acts=mean_acts)

interv_results = {}
for cfg_name, skip in tqdm(INTERV_CONFIGS.items(), desc="Interventions"):
    loader_iv = make_loader(dataset, batch_size=32, shuffle=False)
    answers = {}
    with torch.no_grad():
        for bi, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_iv):
            if bi >= MAX_BATCHES_NR:
                break
            Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
            Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)
            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L; ymax = idx % L
            ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers.update(ans)
    metrics = squad_evaluate(eval_file, answers)
    interv_results[cfg_name] = {"f1": metrics["f1"], "em": metrics["exact_match"]}

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_intervention_spectrum.json", "w") as f:
    json.dump(interv_results, f, indent=2)

# ── Results Table ──
print(f"\n{'='*70}")
print("INTERVENTION SPECTRUM RESULTS")
print(f"{'='*70}")

print(f"{'Component':<10s}  {'Zero':>10s}  {'Noise':>10s}  {'Mean':>10s}  {'Baseline':>10s}")

print("-" * 55)
for comp in ["conv_1", "conv_0", "attn", "ffn"]:
    z = interv_results[f"zero_{comp}"]["f1"]
    n = interv_results[f"noise_{comp}"]["f1"]
    m = interv_results[f"mean_{comp}"]["f1"]
    print(f"{comp:<10s}  {z:10.2f}  {n:10.2f}  {m:10.2f}  {nr_base_f1:10.2f}")

print(f"\n{'Component':<10s}  {'Zero ΔF1':>10s}  {'Noise ΔF1':>10s}  {'Mean ΔF1':>10s}")
print("-" * 45)
for comp, label in [("conv_1", "Conv_1"), ("conv_0", "Conv_0"), ("attn", "Attn"), ("ffn", "FFN")]:
    z_d = interv_results[f"zero_{comp}"]["f1"] - nr_base_f1
    n_d = interv_results[f"noise_{comp}"]["f1"] - nr_base_f1
    m_d = interv_results[f"mean_{comp}"]["f1"] - nr_base_f1
    print(f"{label:<10s}  {z_d:+10.2f}  {n_d:+10.2f}  {m_d:+10.2f}")


In [ ]:
## H1 — Linear Probe: What information does Conv_1 encode?
##
## Train a linear classifier on each sub-layer's per-token output:
##   label = 1 if token is inside the answer span, 0 otherwise.
## Compare probe accuracy across Conv_0 / Conv_1 / Self-Attn / FFN.
## If Conv_1 probe >> Conv_0 probe → Conv_1 encodes answer-relevant features
## that Conv_0 (same architecture) does not.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

N_PROBE = 500
PROBE_BLOCKS = [(0, 0), (0, 3), (0, 6), (1, 0), (2, 0)]  # sample of (pass, block)
loader_probe = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)

comp_names_probe = ["conv_0", "conv_1", "self_attn", "ffn"]

# Collect features: {(pass, block, comp): {"X": [...], "y": [...]}}
probe_data = {}
for pb in PROBE_BLOCKS:
    for c in comp_names_probe:
        probe_data[(pb[0], pb[1], c)] = {"X": [], "y": []}

samples_probe = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_probe, total=N_PROBE, desc="Probe features"):
        if samples_probe >= N_PROBE:
            break
        y1_val, y2_val = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1_val >= L or y2_val >= L or y1_val > y2_val:
            continue

        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)

        # Labels: 1 for answer tokens, 0 for non-answer (only non-padding)
        pad_mask = (Cwid[0].cpu() != 0).numpy()  # True for real tokens
        labels = np.zeros(L, dtype=int)
        labels[y1_val:y2_val+1] = 1

        model_enc_acts = clean_acts.get("model_enc", {})
        for (pi, bi) in PROBE_BLOCKS:
            blk_acts = model_enc_acts.get(f"pass_{pi}", {}).get(f"block_{bi}", {})
            for c in comp_names_probe:
                act = blk_acts.get(c)
                if act is None:
                    continue
                feat = act[0].cpu().float().numpy()  # [d_model, L]
                for t in range(L):
                    if not pad_mask[t]:
                        continue
                    probe_data[(pi, bi, c)]["X"].append(feat[:, t])
                    probe_data[(pi, bi, c)]["y"].append(labels[t])

        samples_probe += 1

# Train probes
print(f"\n{'='*70}")
print(f"{'='*70}")

probe_results = {}
print(f"{'(pass,block)':<14s}  {'Component':<12s}  {'AUC':>6s}  {'Acc':>6s}  {'F1(ans)':>8s}  {'N_train':>8s}  {'%pos':>6s}")
print("-" * 70)

for (pi, bi) in PROBE_BLOCKS:
    for c in comp_names_probe:
        key = (pi, bi, c)
        X = np.array(probe_data[key]["X"])
        y = np.array(probe_data[key]["y"])
        if len(X) < 100 or y.sum() < 10:
            continue

        # Train/test split (deterministic)
        n = len(X)
        idx = np.arange(n)
        np.random.seed(42)
        np.random.shuffle(idx)
        split = int(0.8 * n)
        X_tr, X_te = X[idx[:split]], X[idx[split:]]
        y_tr, y_te = y[idx[:split]], y[idx[split:]]

        clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)
        y_prob = clf.predict_proba(X_te)[:, 1]

        auc = roc_auc_score(y_te, y_prob)
        acc = accuracy_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred)
        pct_pos = y.mean() * 100

        probe_results[key] = {"auc": auc, "acc": acc, "f1_ans": f1, "n": len(X), "pct_pos": pct_pos}
        print(f"p{pi}_b{bi}        {c:<12s}  {auc:6.3f}  {acc:6.3f}  {f1:8.3f}  {len(X):8d}  {pct_pos:5.1f}%")
    print()

# Summary: average AUC by component type
print(f"--- Average AUC by Component Type (across all probed blocks) ---")
for c in comp_names_probe:
    aucs = [v["auc"] for k, v in probe_results.items() if k[2] == c]
    if aucs:
        print(f"  {c:<14s}: mean AUC = {np.mean(aucs):.3f}  (n={len(aucs)} blocks)")

print(f"\n--- Average AUC by Block Location ---")
for (pi, bi) in PROBE_BLOCKS:
    aucs = [v["auc"] for k, v in probe_results.items() if k[0] == pi and k[1] == bi]
    if aucs:
        print(f"  p{pi}_b{bi}: mean AUC = {np.mean(aucs):.3f}")

# The critical comparison
conv1_aucs = [v["auc"] for k, v in probe_results.items() if k[2] == "conv_1"]
conv0_aucs = [v["auc"] for k, v in probe_results.items() if k[2] == "conv_0"]
attn_aucs = [v["auc"] for k, v in probe_results.items() if k[2] == "self_attn"]
if conv1_aucs and conv0_aucs:
    else:

os.makedirs("experiments/results/H1", exist_ok=True)
probe_save = {f"p{k[0]}_b{k[1]}_{k[2]}": v for k, v in probe_results.items()}
with open("experiments/results/H1/h1_linear_probe.json", "w") as f:
    json.dump(probe_save, f, indent=2)

In [ ]:
## H1 — Experiment 2c: Local Coherence (Spatial Autocorrelation)
##
## Measures mean cosine similarity between output vectors at positions t and t+lag.
## Quantifies the spatial correlation structure each component imposes.
## Corresponds to report Experiment 2c.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
import numpy as np

N_COH = 500
comp_names_coh = ["conv_0", "conv_1", "self_attn", "ffn"]
coherence = {c: {lag: [] for lag in [1, 2, 3, 5]} for c in comp_names_coh}

loader_coh = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(42); torch.manual_seed(42)

n_coh = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_coh, total=N_COH, desc="Exp2c: Local Coherence"):
        if n_coh >= N_COH:
            break
        L = Cwid.size(1)
        if y1.item() >= L or y2.item() >= L:
            continue

        Cwid_d, Ccid_d = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid_d, Qcid_d = Qwid.to(DEVICE), Qcid.to(DEVICE)
        _, _, clean_acts, _ = qanet_forward(model, Cwid_d, Ccid_d, Qwid_d, Qcid_d, collect=True)

        pad_mask = (Cwid[0].cpu() != 0).numpy()
        me = clean_acts.get("model_enc", {})
        ba = me.get("pass_0", {}).get("block_0", {})

        for c in comp_names_coh:
            act = ba.get(c)
            if act is None:
                continue
            feat = act[0].cpu().float().numpy()
            nrm = np.linalg.norm(feat, axis=0)
            ok = pad_mask & (nrm > 1e-8)
            for lag in [1, 2, 3, 5]:
                for t in range(L - lag):
                    if ok[t] and ok[t + lag]:
                        cs = np.dot(feat[:, t], feat[:, t + lag]) / (nrm[t] * nrm[t + lag])
                        coherence[c][lag].append(cs)
        n_coh += 1

print(f"\n{'='*80}")
print(f"H1 Experiment 2c: Local Coherence — Spatial Autocorrelation (p0_b0)")
print(f"{'='*80}")
print(f"Samples: {n_coh}")
print(f"Metric: mean cosine similarity between output vectors at positions t and t+lag\n")

print(f"{'Component':<14s}  {'lag=1':>8s}  {'lag=2':>8s}  {'lag=3':>8s}  {'lag=5':>8s}  {'decay(1→5)':>11s}")
print("-" * 60)
coh_summary = {}
for c in comp_names_coh:
    v = {}
    for lag in [1, 2, 3, 5]:
        v[lag] = np.mean(coherence[c][lag]) if coherence[c][lag] else 0
    decay = v[1] - v[5]
    coh_summary[c] = v
    print(f"{c:<14s}  {v[1]:8.4f}  {v[2]:8.4f}  {v[3]:8.4f}  {v[5]:8.4f}  {decay:11.4f}")

print(f"\nInterpretation:")
print(f"  Higher lag-1 = more locally coherent output")
print(f"  Larger decay(1->5) = features are spatially structured (local, not global)")

os.makedirs("experiments/results/H1", exist_ok=True)
coh_save = {c: {str(lag): float(np.mean(vs)) if vs else 0.0 for lag, vs in d.items()} for c, d in coherence.items()}
with open("experiments/results/H1/h1_local_coherence.json", "w") as f:
    json.dump(coh_save, f, indent=2)
print(f"Results saved to experiments/results/H1/h1_local_coherence.json")

In [ ]:
## H1 — Attention Pattern Degradation: Positive evidence for Conv_1's role
##
## Core idea: If Conv_1 is the "local feature substrate" for Self-Attention,
## removing Conv_1 should degrade attention patterns (higher entropy, lost focus).
##
## Conditions:  clean  |  -Conv_1 (critical)  |  -Conv_0 (control)
## Metrics per sample, per block:
##   1. Attention entropy  (higher = more uniform = degraded)
##   2. JS divergence vs clean  (how much the pattern shifted)
##   3. Attention mass on answer-span tokens  (focus lost?)
##
## Memory-efficient: compute metrics on-the-fly per sample; only keep stats.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, SkipSpec
from Models.encoder import mask_logits

print("=" * 70)
print("=" * 70)

# ── Metric helpers (operate on single attention tensors) ──
def _entropy(attn):
    """attn: [1, H, L, L] → scalar mean entropy."""
    eps = 1e-10
    return -(attn * (attn + eps).log()).sum(dim=-1).mean().item()

def _js_div(p, q):
    """JS divergence between two [1, H, L, L] attention tensors."""
    eps = 1e-10
    m = 0.5 * (p + q)
    kl_pm = (p * ((p + eps) / (m + eps)).log()).sum(dim=-1).mean()
    kl_qm = (q * ((q + eps) / (m + eps)).log()).sum(dim=-1).mean()
    return (0.5 * (kl_pm + kl_qm)).item()

def _ans_mass(attn, s, e):
    """Fraction of attention on answer span [s, e]. attn: [1, H, L, L]."""
    if s > e or e >= attn.shape[-1]:
        return float('nan')
    return attn[:, :, :, s:e+1].sum(dim=-1).mean().item()

# ── Hook: recompute attention weights from Q,K inside self_att ──
_attn_store = {}

def _make_hook(name):
    def hook(module, inputs, output):
        x, mask = inputs[0], inputs[1]
        sa = module
        B, C, L = x.size()
        x_t = x.transpose(1, 2)
        q = sa.q_linear(x_t).view(B, L, sa.num_heads, sa.d_k)
        k = sa.k_linear(x_t).view(B, L, sa.num_heads, sa.d_k)
        q = q.permute(2, 0, 1, 3).contiguous().view(B * sa.num_heads, L, sa.d_k)
        k = k.permute(2, 0, 1, 3).contiguous().view(B * sa.num_heads, L, sa.d_k)
        m = mask.bool() if mask.dtype != torch.bool else mask
        am = m.unsqueeze(1).expand(-1, L, -1).repeat(sa.num_heads, 1, 1)
        scores = torch.bmm(q, k.transpose(1, 2)) * sa.scale
        scores = mask_logits(scores, am)
        attn = F.softmax(scores, dim=2)
        _attn_store[name] = attn.view(sa.num_heads, B, L, L).permute(1, 0, 2, 3).detach()
    return hook

handles = []
for bi, blk in enumerate(model.model_enc_blks):
    handles.append(blk.self_att.register_forward_hook(_make_hook(f"b{bi}")))
print(f"Registered hooks on {len(handles)} encoder blocks.\n")

# ── Main loop: per-sample, compute metrics on-the-fly ──
N_SAMPLES = 50
CONDITIONS = {
    "clean":       None,
    "skip_conv_1": SkipSpec(global_skip={"conv_1"}, mode="zero"),
    "skip_conv_0": SkipSpec(global_skip={"conv_0"}, mode="zero"),
}
COND_NAMES = list(CONDITIONS.keys())

ent_acc  = {c: [[] for _ in range(7)] for c in COND_NAMES}
jsd_acc  = {c: [[] for _ in range(7)] for c in ["skip_conv_1", "skip_conv_0"]}
am_acc   = {c: [[] for _ in range(7)] for c in COND_NAMES}

loader_ad = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(123); torch.manual_seed(123)

sample_count = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_ad, total=N_SAMPLES, desc="Attn degradation"):
        if sample_count >= N_SAMPLES:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        ans_s, ans_e = y1.item(), y2.item()

        sample_attn = {}
        for cond_name, skip in CONDITIONS.items():
            _attn_store.clear()
            qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)
            sample_attn[cond_name] = {f"b{bi}": _attn_store[f"b{bi}"].cpu() for bi in range(7) if f"b{bi}" in _attn_store}

        for bi in range(7):
            key = f"b{bi}"
            a_clean = sample_attn["clean"].get(key)
            a_c1    = sample_attn["skip_conv_1"].get(key)
            a_c0    = sample_attn["skip_conv_0"].get(key)
            if a_clean is None:
                continue

            ent_acc["clean"][bi].append(_entropy(a_clean))
            ent_acc["skip_conv_1"][bi].append(_entropy(a_c1))
            ent_acc["skip_conv_0"][bi].append(_entropy(a_c0))

            jsd_acc["skip_conv_1"][bi].append(_js_div(a_clean, a_c1))
            jsd_acc["skip_conv_0"][bi].append(_js_div(a_clean, a_c0))

            m = _ans_mass(a_clean, ans_s, ans_e)
            if not np.isnan(m):
                am_acc["clean"][bi].append(m)
                am_acc["skip_conv_1"][bi].append(_ans_mass(a_c1, ans_s, ans_e))
                am_acc["skip_conv_0"][bi].append(_ans_mass(a_c0, ans_s, ans_e))

        del sample_attn
        sample_count += 1

for h in handles:
    h.remove()
print(f"Processed {sample_count} samples. Hooks removed.\n")

# ── Results tables ──
print(f"{'='*70}")
print("ATTENTION PATTERN ANALYSIS")
print(f"{'='*70}")

print(f"\n--- Attention Entropy (higher = more uniform = degraded) ---")
print(f"{'Block':<8s}  {'Clean':>8s}  {'-Conv_1':>8s}  {'-Conv_0':>8s}  {'dE(-C1)':>8s}  {'dE(-C0)':>8s}")
print("-" * 50)
ent_means = {c: [] for c in COND_NAMES}
for bi in range(7):
    row = {}
    for c in COND_NAMES:
        row[c] = np.mean(ent_acc[c][bi]) if ent_acc[c][bi] else 0
        ent_means[c].append(row[c])
    dc1 = row["skip_conv_1"] - row["clean"]
    dc0 = row["skip_conv_0"] - row["clean"]
    print(f"Block {bi}   {row['clean']:8.3f}  {row['skip_conv_1']:8.3f}  {row['skip_conv_0']:8.3f}  {dc1:+8.3f}  {dc0:+8.3f}")
avg_ent = {c: np.mean(ent_means[c]) for c in COND_NAMES}
avg_dc1 = avg_ent["skip_conv_1"] - avg_ent["clean"]
avg_dc0 = avg_ent["skip_conv_0"] - avg_ent["clean"]
print(f"{'Avg':<8s}  {avg_ent['clean']:8.3f}  {avg_ent['skip_conv_1']:8.3f}  {avg_ent['skip_conv_0']:8.3f}  {avg_dc1:+8.3f}  {avg_dc0:+8.3f}")

print(f"\n--- JS Divergence (clean vs ablated) ---")
print(f"{'Block':<8s}  {'JSD(-C1)':>12s}  {'JSD(-C0)':>12s}  {'C1/C0':>8s}")
print("-" * 45)
jsd_means = {c: [] for c in ["skip_conv_1", "skip_conv_0"]}
for bi in range(7):
    j1 = np.mean(jsd_acc["skip_conv_1"][bi]) if jsd_acc["skip_conv_1"][bi] else 0
    j0 = np.mean(jsd_acc["skip_conv_0"][bi]) if jsd_acc["skip_conv_0"][bi] else 0
    jsd_means["skip_conv_1"].append(j1)
    jsd_means["skip_conv_0"].append(j0)
    r = j1 / max(j0, 1e-8)
    print(f"Block {bi}   {j1:12.4f}  {j0:12.4f}  {r:7.1f}x")
avg_j1 = np.mean(jsd_means["skip_conv_1"]); avg_j0 = np.mean(jsd_means["skip_conv_0"])
print(f"{'Avg':<8s}  {avg_j1:12.4f}  {avg_j0:12.4f}  {avg_j1/max(avg_j0,1e-8):7.1f}x")

print(f"\n--- Answer-Span Attention Mass ---")
print(f"{'Block':<8s}  {'Clean':>8s}  {'-Conv_1':>8s}  {'-Conv_0':>8s}  {'d(-C1)':>8s}  {'d(-C0)':>8s}")
print("-" * 50)
am_means = {c: [] for c in COND_NAMES}
for bi in range(7):
    row = {}
    for c in COND_NAMES:
        row[c] = np.mean(am_acc[c][bi]) if am_acc[c][bi] else 0
        am_means[c].append(row[c])
    dc1 = row["skip_conv_1"] - row["clean"]
    dc0 = row["skip_conv_0"] - row["clean"]
    print(f"Block {bi}   {row['clean']:8.4f}  {row['skip_conv_1']:8.4f}  {row['skip_conv_0']:8.4f}  {dc1:+8.4f}  {dc0:+8.4f}")
avg_am = {c: np.mean(am_means[c]) for c in COND_NAMES}
avg_am_dc1 = avg_am["skip_conv_1"] - avg_am["clean"]
avg_am_dc0 = avg_am["skip_conv_0"] - avg_am["clean"]
print(f"{'Avg':<8s}  {avg_am['clean']:8.4f}  {avg_am['skip_conv_1']:8.4f}  {avg_am['skip_conv_0']:8.4f}  {avg_am_dc1:+8.4f}  {avg_am_dc0:+8.4f}")

# ── Save ──
attn_results = {
    "n_samples": sample_count,
    "entropy": {"clean": ent_means["clean"], "skip_conv_1": ent_means["skip_conv_1"],
                "skip_conv_0": ent_means["skip_conv_0"], "avg_delta_c1": avg_dc1, "avg_delta_c0": avg_dc0},
    "js_divergence": {"skip_conv_1": jsd_means["skip_conv_1"], "skip_conv_0": jsd_means["skip_conv_0"]},
    "answer_mass": {"clean": am_means["clean"], "skip_conv_1": am_means["skip_conv_1"],
                    "skip_conv_0": am_means["skip_conv_0"]},
}
os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_attn_degradation.json", "w") as f:
    json.dump(attn_results, f, indent=2)


In [ ]:
## H1 — Representational Discriminativity: What Conv_1 ACTUALLY produces
##
## Direct measurement of Conv_1's output properties vs Conv_0 (matched control).
## If Conv_1 produces more discriminative representations (higher token-to-token
## variance, lower adjacent-token similarity, higher effective rank), this directly
## explains WHY attention depends on it.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward

print("=" * 70)
print("=" * 70)

N_DISC = 200
comp_names = ["conv_0", "conv_1", "self_attn", "ffn"]
PROBE_LOCS = [(0, 0), (0, 3), (0, 6), (1, 0), (2, 0)]

# Accumulators: {(pass, block, comp): list of per-sample metric values}
token_var_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}
local_contrast_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}
eff_rank_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}
norm_std_acc = {c: {loc: [] for loc in PROBE_LOCS} for c in comp_names}

loader_disc = make_loader(dataset, batch_size=1, shuffle=True)
random.seed(77); torch.manual_seed(77)

n_done = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_disc, total=N_DISC, desc="Discriminativity"):
        if n_done >= N_DISC:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        cmask = (Cwid == 0).squeeze(0)
        valid = (~cmask).sum().item()

        model_enc = acts.get("model_enc", {})
        for (pi, bi) in PROBE_LOCS:
            blk = model_enc.get(f"pass_{pi}", {}).get(f"block_{bi}", {})
            for c in comp_names:
                out = blk.get(c)
                if out is None:
                    continue
                # out: [1, d_model, L] → [L, d_model] (valid tokens only)
                v = out[0, :, :valid].transpose(0, 1).float()  # [L_valid, d]

                # 1. Token variance: var of L2 norms across positions
                norms = v.norm(dim=1)  # [L_valid]
                token_var_acc[c][(pi, bi)].append(norms.var().item())

                # 2. Norm std (normalized): std/mean of token norms
                mn = norms.mean().item()
                norm_std_acc[c][(pi, bi)].append(norms.std().item() / max(mn, 1e-8))

                # 3. Local contrast: 1 - avg cosine(t, t+1)
                if v.shape[0] > 1:
                    cos = F.cosine_similarity(v[:-1], v[1:], dim=1)  # [L-1]
                    local_contrast_acc[c][(pi, bi)].append((1.0 - cos.mean()).item())

                # 4. Effective rank (via singular values)
                try:
                    S = torch.linalg.svdvals(v)  # [min(L, d)]
                    S = S / S.sum().clamp(min=1e-8)
                    ent = -(S * (S + 1e-10).log()).sum()
                    eff_rank_acc[c][(pi, bi)].append(ent.exp().item())
                except:
                    pass

        n_done += 1

print(f"\nProcessed {n_done} samples.\n")

# ── Results ──
print(f"{'='*70}")
print("REPRESENTATIONAL DISCRIMINATIVITY RESULTS")
print(f"{'='*70}")

for metric_name, acc, higher_is in [
    ("Token Norm Variance", token_var_acc, "more discriminative"),
    ("Normalized Norm Std (CoV)", norm_std_acc, "more discriminative"),
    ("Local Contrast (1−cos)", local_contrast_acc, "more discriminative"),
    ("Effective Rank", eff_rank_acc, "more information-rich"),
]:
    print(f"\n--- {metric_name} (higher = {higher_is}) ---")
    print(f"{'Location':<10s}", end="")
    for c in comp_names:
        print(f"  {c:>12s}", end="")
    print(f"  {'C1/C0':>8s}")
    print("-" * 72)
    c1_all, c0_all = [], []
    for loc in PROBE_LOCS:
        print(f"p{loc[0]}_b{loc[1]:<6d}", end="")
        vals = {}
        for c in comp_names:
            v = np.mean(acc[c][loc]) if acc[c][loc] else 0
            vals[c] = v
            print(f"  {v:12.4f}", end="")
        ratio = vals["conv_1"] / max(vals["conv_0"], 1e-8)
        print(f"  {ratio:7.1f}x")
        c1_all.append(vals["conv_1"])
        c0_all.append(vals["conv_0"])
    avg_c1 = np.mean(c1_all); avg_c0 = np.mean(c0_all)
    print(f"{'Avg':<10s}", end="")
    for c in comp_names:
        avg = np.mean([np.mean(acc[c][loc]) if acc[c][loc] else 0 for loc in PROBE_LOCS])
        print(f"  {avg:12.4f}", end="")
    print(f"  {avg_c1/max(avg_c0,1e-8):7.1f}x")

# ── Component ranking ──
print(f"\n{'='*70}")
print("COMPONENT RANKING BY DISCRIMINATIVITY (averaged across locations)")
print(f"{'='*70}")
print(f"{'Metric':<28s}  {'Conv_0':>8s}  {'Conv_1':>8s}  {'Attn':>8s}  {'FFN':>8s}  {'Winner':<10s}")
print("-" * 80)
for metric_name, acc in [
    ("Token Norm Variance", token_var_acc),
    ("Norm CoV", norm_std_acc),
    ("Local Contrast", local_contrast_acc),
    ("Effective Rank", eff_rank_acc),
]:
    avgs = {}
    for c in comp_names:
        avgs[c] = np.mean([np.mean(acc[c][loc]) if acc[c][loc] else 0 for loc in PROBE_LOCS])
    winner = max(avgs, key=avgs.get)
    print(f"{metric_name:<28s}  {avgs['conv_0']:8.4f}  {avgs['conv_1']:8.4f}  {avgs['self_attn']:8.4f}  {avgs['ffn']:8.4f}  {winner}")

# ── Save ──
disc_results = {}
for metric_name, acc in [("token_var", token_var_acc), ("norm_cov", norm_std_acc),
                          ("local_contrast", local_contrast_acc), ("eff_rank", eff_rank_acc)]:
    disc_results[metric_name] = {}
    for c in comp_names:
        disc_results[metric_name][c] = np.mean([np.mean(acc[c][loc]) if acc[c][loc] else 0 for loc in PROBE_LOCS])

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_discriminativity.json", "w") as f:
    json.dump(disc_results, f, indent=2)


In [ ]:
## H1 Method 1 — Eval-time Global Ablation
## Skip ALL conv / ALL attn / ALL ffn across Model Encoder, measure F1/EM on full dev set.
##
## Implementation: zero-out the sub-layer output BEFORE residual add.
## Effect: only the residual (skip connection) survives → removes that component's computation.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, SkipSpec
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate

print("=" * 70)
print("H1 METHOD 1: EVAL-TIME GLOBAL ABLATION")
print("=" * 70)
print(f"Eval on full dev set ({len(dataset)} samples), batch_size=32.\n")

ABLATION_CONFIGS = {
    "baseline":    None,
    "skip_conv":   SkipSpec(global_skip={"conv"}),
    "skip_conv_0": SkipSpec(global_skip={"conv_0"}),
    "skip_conv_1": SkipSpec(global_skip={"conv_1"}),
    "skip_attn":   SkipSpec(global_skip={"self_attn"}),
    "skip_ffn":    SkipSpec(global_skip={"ffn"}),
}

ablation_results = {}
for cfg_name, skip in ABLATION_CONFIGS.items():
    loader_abl = make_loader(dataset, batch_size=32, shuffle=False)
    answers = {}
    with torch.no_grad():
        for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_abl, desc=f"  {cfg_name}"):
            Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
            Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)

            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L; ymax = idx % L
            ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers.update(ans)

    metrics = squad_evaluate(eval_file, answers)
    ablation_results[cfg_name] = {"f1": metrics["f1"], "em": metrics["exact_match"]}

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_ablation_global.json", "w") as f:
    json.dump(ablation_results, f, indent=2)

base_f1 = ablation_results["baseline"]["f1"]
base_em = ablation_results["baseline"]["em"]

print(f"\n{'='*70}")
print(f"{'Config':>14s}  {'F1':>7s}  {'EM':>7s}  {'ΔF1':>8s}  {'ΔEM':>8s}  {'%F1 lost':>9s}")
print("-" * 70)
print(f"{'baseline':>14s}  {base_f1:7.2f}  {base_em:7.2f}  {'—':>8s}  {'—':>8s}  {'—':>9s}")
for k in ["skip_conv", "skip_conv_0", "skip_conv_1", "skip_attn", "skip_ffn"]:
    r = ablation_results[k]
    df1 = r["f1"] - base_f1
    dem = r["em"] - base_em
    pct = abs(df1) / base_f1 * 100
    print(f"{k:>14s}  {r['f1']:7.2f}  {r['em']:7.2f}  {df1:+8.2f}  {dem:+8.2f}  {pct:8.1f}%")
print("-" * 70)

conv_drop  = base_f1 - ablation_results["skip_conv"]["f1"]
conv0_drop = base_f1 - ablation_results["skip_conv_0"]["f1"]
conv1_drop = base_f1 - ablation_results["skip_conv_1"]["f1"]
attn_drop  = base_f1 - ablation_results["skip_attn"]["f1"]
ffn_drop   = base_f1 - ablation_results["skip_ffn"]["f1"]

print(f"\n--- Damage Ranking ---")
ranking = sorted([
    ("skip_conv", conv_drop), ("skip_conv_0", conv0_drop), ("skip_conv_1", conv1_drop),
    ("skip_attn", attn_drop), ("skip_ffn", ffn_drop),
], key=lambda x: -x[1])
for i, (name, drop) in enumerate(ranking, 1):
    print(f"  {i}. {name:<14s}  −{drop:.2f} F1")

print(f"\n--- Conv_0 vs Conv_1 Analysis ---")
print(f"  skip_conv_0: ΔF1 = {-conv0_drop:+.2f}  (21 layers zeroed)")
print(f"  skip_conv_1: ΔF1 = {-conv1_drop:+.2f}  (21 layers zeroed)")
print(f"  Conv_1/Conv_0 damage ratio: {conv1_drop/max(conv0_drop, 0.01):.2f}x")
print(f"  skip_conv (both): ΔF1 = {-conv_drop:+.2f}")
print(f"  Additivity check: conv0_drop + conv1_drop = {conv0_drop+conv1_drop:.2f} vs conv_both_drop = {conv_drop:.2f}")
interaction = conv_drop - (conv0_drop + conv1_drop)

print(f"\n--- Conv vs Attn Overall ---")
print(f"  Conv(all)/Attn damage ratio: {conv_drop/max(attn_drop, 0.01):.2f}x")
print(f"  Conv_1 alone vs Attn: {conv1_drop/max(attn_drop, 0.01):.2f}x")

In [ ]:
## H1 Method 3 — Per-block Ablation
## Skip conv or attn in ONE specific (pass, block) at a time, measure F1.
## 7 blocks × 3 passes × 2 types = 42 configs.
## "conv" skip zeroes conv_0 AND conv_1 together within one block.

MAX_BATCHES_M3 = None  # None = full dev set (10465 samples, ~1.5 min/config, ~63 min total)

n_samples_est = min(len(dataset), MAX_BATCHES_M3 * 32) if MAX_BATCHES_M3 else len(dataset)
print("=" * 70)
print("H1 METHOD 3: PER-BLOCK ABLATION")
print("=" * 70)
print(f"Total configs: 7 blocks × 3 passes × 2 types = 42")
print(f"Eval on {'~' + str(n_samples_est) + ' subsampled' if MAX_BATCHES_M3 else 'full ' + str(len(dataset))} samples, batch_size=32.")
print(f"Baseline F1 = {base_f1:.2f}, EM = {base_em:.2f}")
if MAX_BATCHES_M3:
    print(f"NOTE: Using {MAX_BATCHES_M3} batches/config for speed. F1 variance ~±1 vs full eval.")
    print("Running subsampled baseline first...\n")

# Run baseline on the SAME subset so ΔF1 is fair
loader_base = make_loader(dataset, batch_size=32, shuffle=False)
answers_base = {}
with torch.no_grad():
    for batch_i, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_base):
        if MAX_BATCHES_M3 and batch_i >= MAX_BATCHES_M3:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=None)
        p1_log = F.log_softmax(p1, dim=1)
        p2_log = F.log_softmax(p2, dim=1)
        outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
        mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
        outer = outer.masked_fill(~mask_tri, float('-inf'))
        flat = outer.view(outer.size(0), -1)
        idx = torch.argmax(flat, dim=1)
        L = p1.size(1)
        ymin = idx // L; ymax = idx % L
        ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
        answers_base.update(ans)
m3_base = squad_evaluate(eval_file, answers_base)
m3_base_f1 = m3_base["f1"]
m3_base_em = m3_base["exact_match"]
print(f"Subsampled baseline ({len(answers_base)} samples): F1 = {m3_base_f1:.2f}, EM = {m3_base_em:.2f}")
print(f"Full dev baseline (Method 1):                      F1 = {base_f1:.2f}, EM = {base_em:.2f}")
print(f"Subset-vs-full difference: ΔF1 = {m3_base_f1 - base_f1:+.2f}\n")

block_ablation = {}
total_configs = 42

for pass_idx in range(3):
    for block_idx in range(7):
        for comp_type in ["conv", "self_attn"]:
            key = f"p{pass_idx}_b{block_idx}_{comp_type}"
            skip = SkipSpec(block_skip=(pass_idx, block_idx, comp_type))
            loader_ba = make_loader(dataset, batch_size=32, shuffle=False)
            answers = {}
            with torch.no_grad():
                for batch_i, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_ba):
                    if MAX_BATCHES_M3 and batch_i >= MAX_BATCHES_M3:
                        break
                    Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
                    Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

                    p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, skip_spec=skip)

                    p1_log = F.log_softmax(p1, dim=1)
                    p2_log = F.log_softmax(p2, dim=1)
                    outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
                    mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1), device=DEVICE, dtype=torch.bool))
                    outer = outer.masked_fill(~mask_tri, float('-inf'))
                    flat = outer.view(outer.size(0), -1)
                    idx = torch.argmax(flat, dim=1)
                    L = p1.size(1)
                    ymin = idx // L; ymax = idx % L
                    ans, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
                    answers.update(ans)

            metrics = squad_evaluate(eval_file, answers)
            m_f1 = metrics["f1"]
            m_em = metrics["exact_match"]
            block_ablation[key] = {"f1": m_f1, "em": m_em}

            done = len(block_ablation)
            df1 = m_f1 - m3_base_f1
            dem = m_em - m3_base_em
            print(f"  [{done:>2d}/{total_configs}] {key:<25s} F1={m_f1:6.2f} (ΔF1={df1:+6.2f})  EM={m_em:6.2f} (ΔEM={dem:+6.2f})")

with open("experiments/results/H1/h1_ablation_perblock.json", "w") as f:
    json.dump({
        "baseline": {"f1": m3_base_f1, "em": m3_base_em, "n_samples": len(answers_base)},
        "configs": block_ablation,
    }, f, indent=2)

ref_f1 = m3_base_f1
ref_em = m3_base_em

print(f"\n{'='*70}")
print(f"PER-BLOCK ABLATION SUMMARY  (subsampled baseline F1={ref_f1:.2f}, EM={ref_em:.2f})")
print(f"{'='*70}")

print(f"\n--- Top 10 most damaging single-block ablations ---")
print(f"{'Rank':<5s} {'Config':<25s}  {'F1':>7s}  {'ΔF1':>8s}  {'EM':>7s}  {'ΔEM':>8s}")
print("-" * 65)
ranked_abl = sorted(block_ablation.items(), key=lambda x: x[1]["f1"])
for rank, (k, r) in enumerate(ranked_abl[:10], 1):
    print(f"{rank:<5d} {k:<25s}  {r['f1']:7.2f}  {r['f1']-ref_f1:+8.2f}  {r['em']:7.2f}  {r['em']-ref_em:+8.2f}")

print(f"\n--- Average ΔF1 by Component Type ---")
conv_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if k.endswith("_conv")]
attn_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if k.endswith("_self_attn")]
print(f"  Skip Conv:      mean ΔF1 = {-np.mean(conv_drops):+.2f}, max drop = {np.max(conv_drops):.2f}, min drop = {np.min(conv_drops):.2f}")
print(f"  Skip Self-Attn: mean ΔF1 = {-np.mean(attn_drops):+.2f}, max drop = {np.max(attn_drops):.2f}, min drop = {np.min(attn_drops):.2f}")

print(f"\n--- Average ΔF1 by Pass ---")
for p in range(3):
    p_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if k.startswith(f"p{p}_")]
    print(f"  Pass {p} → M{p+1}: mean ΔF1 = {-np.mean(p_drops):+.2f}, max drop = {np.max(p_drops):.2f}")

print(f"\n--- Average ΔF1 by Block Depth ---")
for b in range(7):
    b_drops = [ref_f1 - r["f1"] for k, r in block_ablation.items() if f"_b{b}_" in k]
    bar = "█" * int(max(np.mean(b_drops), 0) * 10)
    print(f"  Block {b}: mean ΔF1 = {-np.mean(b_drops):+.2f}  {bar}")

In [ ]:
## H1 — Publication Figures
## Generates 3 figures for the report: causal tracing heatmap, dual-method
## cross-validation, and attention JSD degradation chart.
## Requires: h1_results.json, h1_ablation_global.json, h1_attn_degradation.json

import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

_H1_DIR = "experiments/results/H1"
_PAL = {"conv1": "#d32f2f", "conv0": "#1976d2", "attn": "#2e7d32",
        "ffn": "#ef6c00", "accent": "#7b1fa2"}
_ORD = ["conv_0", "conv_1", "self_attn", "ffn"]
_LBL = {"conv_0": "Conv₀", "conv_1": "Conv₁", "self_attn": "Self-Attn", "ffn": "FFN"}
_CLR = {"conv_0": _PAL["conv0"], "conv_1": _PAL["conv1"],
        "self_attn": _PAL["attn"], "ffn": _PAL["ffn"]}

def _ld(name):
    p = os.path.join(_H1_DIR, name)
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f)

# ── Fig 1: Causal Tracing Heatmap ────────────────────────────────
ct_data = _ld("h1_results.json")
if ct_data is not None:
    res = ct_data["results"]
    grids = {}
    for comp in _ORD:
        g = np.zeros((3, 7))
        for p in range(3):
            for b in range(7):
                g[p, b] = res.get(f"p{p}_b{b}_{comp}", {}).get("aie_span", 0.0)
        grids[comp] = g

    vmax = max(g.max() for g in grids.values())
    purples = mcolors.LinearSegmentedColormap.from_list("WtPrpl", ["#ffffff", "#4527a0"])
    norm = mcolors.Normalize(vmin=0, vmax=vmax)

    fig = plt.figure(figsize=(13, 6.4), dpi=150, facecolor="white")
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.05], wspace=0.30, hspace=0.45,
                          left=0.10, right=0.88, top=0.88, bottom=0.08)
    fig.suptitle("Causal Tracing: AIE per Sub-layer", fontsize=15, fontweight="bold", y=0.95)
    layout = [(0, 0, "conv_0"), (0, 1, "conv_1"), (1, 0, "self_attn"), (1, 1, "ffn")]
    for r, c, comp in layout:
        ax = fig.add_subplot(gs[r, c])
        ax.set_facecolor("white")
        g = grids[comp]
        im = ax.imshow(g, cmap=purples, norm=norm, aspect="auto", interpolation="nearest")
        ax.set_title(f"{_LBL[comp]}    (Sum AIE = {g.sum():.3f})",
                     fontsize=12, fontweight="bold", color=_CLR[comp])
        ax.set_xticks(range(7))
        ax.set_xticklabels([f"B{i}" for i in range(7)], fontsize=9)
        if r == 1:
            ax.set_xlabel("Block", fontsize=10, fontweight="bold")
        ax.set_yticks(range(3))
        if c == 0:
            ax.set_yticklabels(["Pass 0  (→M₁)", "Pass 1  (→M₂)", "Pass 2  (→M₃)"], fontsize=9)
        else:
            ax.set_yticklabels([])
        for pi in range(3):
            for bi in range(7):
                v = g[pi, bi]
                ax.text(bi, pi, f"{v:.3f}", ha="center", va="center",
                        fontsize=6.5, fontweight="bold", color="white" if v > vmax * 0.45 else "black")
    cax = fig.add_subplot(gs[:, 2])
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_label("AIE  (span-probability recovery)", fontsize=10, fontweight="bold")
    cbar.ax.tick_params(labelsize=8)
    os.makedirs("experiments/prism-uploads", exist_ok=True)
    fig.savefig("experiments/prism-uploads/h1_fig1_causal_tracing_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: experiments/prism-uploads/h1_fig1_causal_tracing_heatmap.png")
else:
    print("[Fig 1] SKIP — h1_results.json not found.  Run the H1 causal tracing cell first.")

# ── Fig 2: Dual-Method Cross-Validation ──────────────────────────
abl_data = _ld("h1_ablation_global.json")
assert abl_data is not None, "h1_ablation_global.json not found — run H1 ablation first"
assert ct_data is not None, "h1_results.json not found — run H1 causal tracing first"

base = abl_data["baseline"]["f1"]
ABL = {}
for comp, key in [("conv_1", "skip_conv_1"), ("conv_0", "skip_conv_0"),
                  ("self_attn", "skip_attn"), ("ffn", "skip_ffn")]:
    ABL[comp] = base - abl_data[key]["f1"]

res = ct_data["results"]
CT = {}
for comp in _ORD:
    CT[comp] = sum(v["aie_span"] for k, v in res.items() if k.endswith(comp))

abl_tot = sum(ABL.values()); ct_tot = sum(CT.values())
abl_pct = {c: ABL[c]/abl_tot*100 for c in _ORD}
ct_pct  = {c: CT[c]/ct_tot*100 for c in _ORD}
x = np.arange(4); w = 0.38

fig, ax = plt.subplots(figsize=(7.5, 4.8), dpi=150)
ax.bar(x - w/2, [abl_pct[c] for c in _ORD], w, label="Ablation  (|ΔF1| share)",
       color=_PAL["conv1"], alpha=0.82, edgecolor="white", linewidth=0.8)
ax.bar(x + w/2, [ct_pct[c] for c in _ORD], w, label="Causal Tracing  (AIE share)",
       color=_PAL["conv0"], alpha=0.82, edgecolor="white", linewidth=0.8)
for i, comp in enumerate(_ORD):
    ax.text(i - w/2, abl_pct[comp]+1.0, f"|ΔF1|={ABL[comp]:.1f}",
            ha="center", fontsize=7.5, fontweight="bold", color="#b71c1c")
    ax.text(i + w/2, ct_pct[comp]+1.0, f"{ct_pct[comp]:.1f}%",
            ha="center", fontsize=7.5, fontweight="bold", color="#0d47a1")
ax.set_ylabel("Share of Total  (%)", fontsize=12, fontweight="bold")
ax.set_title("Cross-Validation: Conv₁ Dominance Confirmed by Both Methods", fontsize=13, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([_LBL[c] for c in _ORD], fontsize=11, fontweight="bold")
ax.legend(fontsize=10, loc="upper right", framealpha=0.92)
ax.set_ylim(0, max(max(abl_pct.values()), max(ct_pct.values())) * 1.15)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
fig.savefig("experiments/prism-uploads/h1_fig2_dual_method.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: experiments/prism-uploads/h1_fig2_dual_method.png")

# ── Fig 3: Attention JSD Degradation ─────────────────────────────
attn = _ld("h1_attn_degradation.json")
assert attn is not None, "h1_attn_degradation.json not found — run H1 attention analysis first"
assert "js_divergence" in attn, "js_divergence key missing in h1_attn_degradation.json"
JSD_C1 = attn["js_divergence"]["skip_conv_1"]
JSD_C0 = attn["js_divergence"]["skip_conv_0"]

x7 = np.arange(7); ratios = [c1/max(c0,1e-12) for c1,c0 in zip(JSD_C1, JSD_C0)]

fig, ax = plt.subplots(figsize=(8, 4.8), dpi=150)
ax.bar(x7 - w/2, JSD_C1, w, label="−Conv₁  (treatment)",
       color=_PAL["conv1"], alpha=0.85, edgecolor="white", linewidth=0.8)
ax.bar(x7 + w/2, JSD_C0, w, label="−Conv₀  (control)",
       color=_PAL["conv0"], alpha=0.85, edgecolor="white", linewidth=0.8)
for i in range(7):
    ax.annotate(f"{ratios[i]:.1f}×", xy=(i, JSD_C1[i]), xytext=(0, 6),
                textcoords="offset points", ha="center",
                fontsize=9, fontweight="bold", color=_PAL["accent"])
avg_c1 = np.mean(JSD_C1); avg_c0 = np.mean(JSD_C0)
ax.axhline(avg_c1, color=_PAL["conv1"], ls="--", lw=1.0, alpha=0.45,
           label=f"−Conv₁ avg ({avg_c1:.4f})")
ax.axhline(avg_c0, color=_PAL["conv0"], ls="--", lw=1.0, alpha=0.45,
           label=f"−Conv₀ avg ({avg_c0:.4f})")
avg_ratio = avg_c1 / max(avg_c0, 1e-12)
ax.text(0.97, 0.94, f"Average ratio:  {avg_ratio:.1f}×\nRange:  {min(ratios):.1f}–{max(ratios):.1f}×",
        transform=ax.transAxes, fontsize=10, fontweight="bold", va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.45", fc="#f3e5f5", ec=_PAL["accent"], alpha=0.92, lw=1.2))
ax.set_xlabel("Encoder Block", fontsize=12, fontweight="bold")
ax.set_ylabel("JS Divergence  (clean vs ablated)", fontsize=12, fontweight="bold")
ax.set_title("Attention Distortion: Removing Conv₁ Is 8× Worse Than Conv₀", fontsize=13, fontweight="bold")
ax.set_xticks(x7); ax.set_xticklabels([f"Block {i}" for i in range(7)], fontsize=10)
ax.legend(fontsize=9, loc="upper left", framealpha=0.92, ncol=2)
ax.set_ylim(0, max(JSD_C1) * 1.25)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
fig.savefig("experiments/prism-uploads/h1_fig3_attention_jsd.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: experiments/prism-uploads/h1_fig3_attention_jsd.png")

### H2: Directional Asymmetry in CQ Attention — Cross-Architecture Validation

**Research question**: BiDAF found C2Q attention 2.7x more important than Q2C (Seo et al., 2017). QANet replaced LSTM with Conv+Attention — does this asymmetry still hold? If so, what mechanism maintains it?

**Experiments**:

1. **Sub-component Ablation**: Zero individual sub-components [C, A, C⊙A, C⊙B] → F1/EM ranking; direction-level aggregation vs BiDAF; cq_resizer weight analysis. Preceded by CQ bottleneck validation (zero entire CQ output).
2. **Information Source Localization**: Selective corruption of answer-span vs non-answer tokens → information density ratio → mechanism explanation.

In [ ]:
## H2 — Experiment 1: Directional Asymmetry — Sub-component Ablation
##
## (a) CQ bottleneck validation: zero entire CQ output → F1/EM
## (b) Sub-component ablation: zero [C], [A], [C⊙A], [C⊙B] → F1/EM ranking
## (c) Direction-level aggregation: C2Q vs Q2C → compare with BiDAF (2.7×)
## (d) cq_resizer weight quadrant analysis

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate
from tqdm.auto import tqdm

print("=" * 70)
print("H2 EXPERIMENT 1: DIRECTIONAL ASYMMETRY — SUB-COMPONENT ABLATION")
print("=" * 70)

# ── (a) CQ Bottleneck Validation ─────────────────────────────────────────
print("\n--- (a) CQ Bottleneck Validation ---")

loader_h2 = make_loader(dataset, batch_size=32, shuffle=False)

bottleneck_configs = {
    "clean":       {"skip_cq_att": False},
    "skip_cq_att": {"skip_cq_att": True},
}

answers_bn = {cfg: {} for cfg in bottleneck_configs}

with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_h2, desc="CQ Bottleneck"):
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        for cfg_name, kwargs in bottleneck_configs.items():
            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, **kwargs)
            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1),
                                             device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~mask_tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L; ymax = idx % L
            preds, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers_bn[cfg_name].update(preds)

metrics_bn = {cfg: squad_evaluate(eval_file, answers_bn[cfg]) for cfg in bottleneck_configs}
base_f1 = metrics_bn["clean"]["f1"]
base_em = metrics_bn["clean"]["exact_match"]
cq_f1_drop = base_f1 - metrics_bn["skip_cq_att"]["f1"]

print(f"  Baseline F1={base_f1:.2f}, EM={base_em:.2f}")
print(f"  CQ zeroed  F1={metrics_bn['skip_cq_att']['f1']:.2f} (drop={cq_f1_drop:.2f}, {cq_f1_drop/base_f1*100:.1f}%)")

# ── (b) Sub-component Ablation ────────────────────────────────────────────
print("\n--- (b) Sub-component Ablation ---")

SUBCOMP_CONFIGS = {
    "baseline":     {"zero_cq_quadrants": None},
    "-C":           {"zero_cq_quadrants": [0]},
    "-A":           {"zero_cq_quadrants": [1]},
    "-C*A":         {"zero_cq_quadrants": [2]},
    "-C*B":         {"zero_cq_quadrants": [3]},
    "-C2Q (A+C*A)": {"zero_cq_quadrants": [1, 2]},
    "-Q2C (C*B)":   {"zero_cq_quadrants": [3]},
    "only C":       {"zero_cq_quadrants": [1, 2, 3]},
}

# Deduplicate configs with same quadrants
seen_keys = set()
unique_configs = {}
for name, cfg in SUBCOMP_CONFIGS.items():
    key = str(cfg.get("zero_cq_quadrants"))
    if key not in seen_keys:
        seen_keys.add(key)
        unique_configs[name] = cfg

loader_h2_sub = make_loader(dataset, batch_size=32, shuffle=False)
answers_sub = {name: {} for name in unique_configs}

with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(loader_h2_sub, desc="Subcomp Ablation"):
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        for cfg_name, kwargs in unique_configs.items():
            p1, p2, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, **kwargs)
            p1_log = F.log_softmax(p1, dim=1)
            p2_log = F.log_softmax(p2, dim=1)
            outer = p1_log.unsqueeze(2) + p2_log.unsqueeze(1)
            mask_tri = torch.triu(torch.ones(outer.size(-2), outer.size(-1),
                                             device=DEVICE, dtype=torch.bool))
            outer = outer.masked_fill(~mask_tri, float('-inf'))
            flat = outer.view(outer.size(0), -1)
            idx = torch.argmax(flat, dim=1)
            L = p1.size(1)
            ymin = idx // L; ymax = idx % L
            preds, _ = convert_tokens(eval_file, ids.tolist(), ymin.tolist(), ymax.tolist())
            answers_sub[cfg_name].update(preds)

metrics_sub = {n: squad_evaluate(eval_file, answers_sub[n]) for n in unique_configs}

def _find_metric(name):
    if name in metrics_sub:
        return metrics_sub[name]
    target = str(SUBCOMP_CONFIGS[name].get("zero_cq_quadrants"))
    for n, c in unique_configs.items():
        if str(c.get("zero_cq_quadrants")) == target:
            return metrics_sub[n]
    return None

base_f1_sub = _find_metric("baseline")["f1"]
base_em_sub = _find_metric("baseline")["exact_match"]

print(f"\n{'Condition':>17s}  {'F1':>7s}  {'EM':>7s}  {'dF1':>8s}  {'dEM':>8s}")
print("-" * 55)
for name in SUBCOMP_CONFIGS:
    m = _find_metric(name)
    if m:
        df1 = m["f1"] - base_f1_sub
        dem = m["exact_match"] - base_em_sub
        print(f"{name:>17s}  {m['f1']:7.2f}  {m['exact_match']:7.2f}  {df1:+8.2f}  {dem:+8.2f}")

# ── (c) Direction-level comparison with BiDAF ─────────────────────────────
print(f"\n--- (c) Direction-level: QANet vs BiDAF ---")
c2q_m = _find_metric("-C2Q (A+C*A)")
q2c_m = _find_metric("-Q2C (C*B)")
c2q_drop = base_f1_sub - c2q_m["f1"] if c2q_m else 0
q2c_drop = base_f1_sub - q2c_m["f1"] if q2c_m else 0
ratio = c2q_drop / max(q2c_drop, 0.01)
print(f"  QANet:  -C2Q = {-c2q_drop:+.2f} F1,  -Q2C = {-q2c_drop:+.2f} F1,  ratio = {ratio:.2f}x")
print(f"  BiDAF:  -C2Q = -9.6 F1,    -Q2C = -3.6 F1,    ratio = 2.67x")

# ── (d) cq_resizer weight analysis ───────────────────────────────────────
print(f"\n--- (d) cq_resizer Weight Analysis ---")
W = model.cq_resizer.weight.data.cpu().squeeze(-1)  # [d, 4d]
d = W.size(0)
quadrant_names = ["C (original)", "A (C2Q)", "C*A (gated)", "C*B (Q2C)"]
total_norm = W.norm().item()
print(f"  {'Quadrant':>15s}  {'Frob. norm':>12s}  {'% of total':>12s}")
print(f"  {'-'*44}")
q_norms = []
for i, qname in enumerate(quadrant_names):
    qn = W[:, i*d:(i+1)*d].norm().item()
    q_norms.append(qn)
    print(f"  {qname:>15s}  {qn:12.4f}  {qn/total_norm*100:11.1f}%")

# ── Save results ──────────────────────────────────────────────────────────
h2_exp1 = {
    "bottleneck": {
        "clean_f1": base_f1, "clean_em": base_em,
        "cq_zeroed_f1": metrics_bn["skip_cq_att"]["f1"],
        "cq_f1_drop": cq_f1_drop,
    },
    "subcomponent": {},
    "cq_resizer_norms": {qname: q_norms[i] for i, qname in enumerate(quadrant_names)},
}
for name in unique_configs:
    m = metrics_sub[name]
    h2_exp1["subcomponent"][name] = {
        "f1": m["f1"], "em": m["exact_match"],
        "df1": m["f1"] - base_f1_sub, "dem": m["exact_match"] - base_em_sub,
        "zero_quadrants": unique_configs[name].get("zero_cq_quadrants"),
    }
h2_exp1["direction_comparison"] = {
    "qanet_c2q_drop": c2q_drop, "qanet_q2c_drop": q2c_drop, "qanet_ratio": ratio,
    "bidaf_c2q_drop": 9.6, "bidaf_q2c_drop": 3.6, "bidaf_ratio": 2.67,
}
os.makedirs("experiments/results/H2", exist_ok=True)
with open("experiments/results/H2/h2_exp1_asymmetry.json", "w") as f:
    json.dump(h2_exp1, f, indent=2)


print(f"\nSaved to experiments/results/H2/h2_exp1_asymmetry.json")

In [ ]:
## H2 — Experiment 2: Information Source Localization
##
## Where in the Context does the critical information live?
## Selective corruption: corrupt only answer-span tokens vs only non-answer tokens.
## Connects back to Exp 2: if C2Q's advantage comes from precise targeting of
## high-density answer positions, then corrupting those positions should cause
## disproportionate damage.

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward, RestoreSpec, compute_span_prob
from tqdm.auto import tqdm

NUM_H2_SEL = 200
NOISE_REPEATS_SEL = 3
SEED = 42
MIN_CLEAN_PROB = 0.01
NOISE_STD_SCALE = 3.0

random.seed(SEED); torch.manual_seed(SEED)
loader_h2_sel = make_loader(dataset, batch_size=1, shuffle=True)

print("=" * 70)
print("H2 EXPERIMENT 3: SELECTIVE CORRUPTION — INFORMATION LOCALIZATION")
print("=" * 70)
print("Conditions:")
print("  ans_only:     corrupt ONLY answer-span positions in context")
print("  non_ans_only: corrupt ONLY non-answer, non-padding positions in context")
print("  full_context: corrupt ALL context positions (baseline from Exp 1)")
print(f"Samples: {NUM_H2_SEL}, Noise repeats: {NOISE_REPEATS_SEL}")
print(f"Also measures CQ-Att IE for each condition (CQ recovery test)\n")

conditions = ["ans_only", "non_ans_only", "full_context"]
sel_te = {c: [] for c in conditions}
sel_cq_ie = {c: [] for c in conditions}
sel_n_corrupted = {c: [] for c in conditions}

cq_restore_spec = RestoreSpec(stage="cq_att", component="output")

samples_sel = 0
pbar_sel = tqdm(total=NUM_H2_SEL, desc="H2 Exp2: Selective Corruption")
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in loader_h2_sel:
        if samples_sel >= NUM_H2_SEL:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        y1_val, y2_val = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1_val >= L or y2_val >= L or y1_val > y2_val:
            continue

        p1_c, p2_c, clean_acts, _ = qanet_forward(
            model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        ans_mask = torch.zeros(1, L, device=DEVICE, dtype=torch.bool)
        ans_mask[0, y1_val:y2_val+1] = True
        non_pad = (Cwid != 0)
        non_ans_mask = (~ans_mask) & non_pad

        n_ans = ans_mask.sum().item()
        n_non_ans = non_ans_mask.sum().item()
        n_total = non_pad.sum().item()

        mask_map = {
            "ans_only": ans_mask,
            "non_ans_only": non_ans_mask,
            "full_context": None,
        }
        n_map = {
            "ans_only": n_ans,
            "non_ans_only": n_non_ans,
            "full_context": n_total,
        }

        for cond in conditions:
            reps_te, reps_cq_ie = [], []
            for rep in range(NOISE_REPEATS_SEL):
                ns = SEED + samples_sel * 100 + rep
                p1_x, p2_x, _, _ = qanet_forward(
                    model, Cwid, Ccid, Qwid, Qcid,
                    corrupt_target="context",
                    noise_std_scale=NOISE_STD_SCALE,
                    noise_seed=ns,
                    corrupt_mask_c=mask_map[cond],
                )
                prob_x = compute_span_prob(p1_x, p2_x, y1, y2)
                te = (prob_clean - prob_x).item()
                reps_te.append(te)

                if abs(te) < 1e-6:
                    continue

                p1_r, p2_r, _, _ = qanet_forward(
                    model, Cwid, Ccid, Qwid, Qcid,
                    corrupt_target="context",
                    noise_std_scale=NOISE_STD_SCALE,
                    noise_seed=ns,
                    corrupt_mask_c=mask_map[cond],
                    clean_acts=clean_acts,
                    restore_spec=cq_restore_spec,
                )
                prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                reps_cq_ie.append((prob_r - prob_x).item())

            sel_te[cond].append(np.mean(reps_te) if reps_te else 0)
            sel_cq_ie[cond].append(np.mean(reps_cq_ie) if reps_cq_ie else 0)
            sel_n_corrupted[cond].append(n_map[cond])

        samples_sel += 1
        pbar_sel.update(1)
        if samples_sel % 50 == 0:
            pbar_sel.set_postfix(
                TE_ans=f"{np.mean(sel_te['ans_only']):.3f}",
                TE_non=f"{np.mean(sel_te['non_ans_only']):.3f}",
                TE_full=f"{np.mean(sel_te['full_context']):.3f}",
            )
pbar_sel.close()

def _ci95(arr):
    a = np.array(arr)
    return 1.96 * a.std() / np.sqrt(len(a)) if len(a) > 1 else 0

print(f"\n{'='*70}")
print("H2 EXP 2: INFORMATION SOURCE LOCALIZATION")
print(f"{'='*70}")
print(f"Samples: {samples_sel}, Noise repeats: {NOISE_REPEATS_SEL}\n")

print(f"--- Total Effect by Corruption Region ---")
print(f"{'Condition':>15s}  {'Mean TE':>10s}  {'±95%CI':>10s}  {'Avg #tokens':>12s}  {'TE/token':>10s}")
print("-" * 70)
for cond in conditions:
    te_arr = np.array(sel_te[cond])
    n_arr = np.array(sel_n_corrupted[cond])
    te_per_tok = te_arr.mean() / max(n_arr.mean(), 1) * 1000
    print(f"{cond:>15s}  {te_arr.mean():10.4f}  {_ci95(te_arr):10.4f}  "
          f"{n_arr.mean():12.1f}  {te_per_tok:10.4f}×10⁻³")

te_ans = np.mean(sel_te["ans_only"])
te_non = np.mean(sel_te["non_ans_only"])
te_full = np.mean(sel_te["full_context"])
n_ans_avg = np.mean(sel_n_corrupted["ans_only"])
n_non_avg = np.mean(sel_n_corrupted["non_ans_only"])

print(f"\n--- Information Localization Analysis ---")
print(f"TE(answer-only)  = {te_ans:.4f}  (avg {n_ans_avg:.1f} tokens)")
print(f"TE(non-answer)   = {te_non:.4f}  (avg {n_non_avg:.1f} tokens)")
print(f"TE(full context) = {te_full:.4f}  (avg {np.mean(sel_n_corrupted['full_context']):.1f} tokens)")
print(f"TE(ans) + TE(non) = {te_ans + te_non:.4f}  vs  TE(full) = {te_full:.4f}")
overlap = te_full - (te_ans + te_non)
print(f"Interaction = {overlap:.4f} ({'sub-additive' if overlap < 0 else 'super-additive'})")

te_per_tok_ans = te_ans / max(n_ans_avg, 1)
te_per_tok_non = te_non / max(n_non_avg, 1)
density_ratio = te_per_tok_ans / max(te_per_tok_non, 1e-8)
print(f"\nInformation density (TE per token):")
print(f"  Answer tokens:     {te_per_tok_ans:.6f}")
print(f"  Non-answer tokens: {te_per_tok_non:.6f}")
print(f"  Density ratio: {density_ratio:.1f}x")
if density_ratio > 3:
elif density_ratio > 1.5:
else:

print(f"\n--- CQ Attention Recovery by Condition ---")
print(f"{'Condition':>15s}  {'Mean CQ IE':>10s}  {'NIE (% of TE)':>15s}")
print("-" * 50)
for cond in conditions:
    cq_ie_mean = np.mean(sel_cq_ie[cond])
    te_mean = np.mean(sel_te[cond])
    nie = cq_ie_mean / max(abs(te_mean), 1e-8) * 100
    print(f"{cond:>15s}  {cq_ie_mean:10.4f}  {nie:14.1f}%")

print(f"If context info is localized to ~{n_ans_avg:.0f} answer tokens but question")
print(f"info is distributed across all ~{Qwid.size(1):.0f} question tokens:")

print(f"\n--- Limitations ---")
print(f"  1. Answer span corruption DIRECTLY removes the target signal — high TE is expected")
print(f"  2. The interesting finding is whether non-answer corruption is near zero or not")
print(f"  3. Noise-based corruption may not perfectly isolate positional information")
print(f"  4. CQ Attention recovery % may vary with corruption locality")

sel_results = {}
for cond in conditions:
    sel_results[cond] = {
        "te_mean": float(np.mean(sel_te[cond])),
        "te_ci95": float(_ci95(sel_te[cond])),
        "cq_ie_mean": float(np.mean(sel_cq_ie[cond])),
        "cq_nie": float(np.mean(sel_cq_ie[cond]) / max(abs(np.mean(sel_te[cond])), 1e-8)),
        "avg_tokens_corrupted": float(np.mean(sel_n_corrupted[cond])),
    }
sel_results["info_density_ratio"] = float(density_ratio)
with open("experiments/results/H2/h2_selective_corruption.json", "w") as f:
    json.dump(sel_results, f, indent=2)
print(f"\nSaved to experiments/results/H2/h2_selective_corruption.json")

In [ ]:
## H2 — Publication Figures
## Fig 1: Cross-architecture direction-level comparison (BiDAF vs QANet)
## Fig 2: Information density — answer vs non-answer TE/token
## Requires: h2_exp1_asymmetry.json, h2_selective_corruption.json

import json, os
import numpy as np
import matplotlib.pyplot as plt

_H2_DIR = "experiments/results/H2"
_PAL = {"c2q": "#d32f2f", "q2c": "#1976d2", "accent": "#7b1fa2", "ans": "#E53935", "non": "#78909C"}

def _ld2(name):
    p = os.path.join(_H2_DIR, name)
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f)

# ── Fig 1: Cross-Architecture Direction-Level Comparison ─────────
BIDF = {"c2q": 9.6, "q2c": 3.6}  # from Seo et al. 2017 Table 3 (reference, not our experiment)

e1 = _ld2("h2_exp1_asymmetry.json")
assert e1 is not None, "h2_exp1_asymmetry.json not found — run H2 Exp 1 first"
assert "direction_comparison" in e1, "direction_comparison key missing in h2_exp1_asymmetry.json"
dc = e1["direction_comparison"]
QANET = {"c2q": abs(dc["qanet_c2q_drop"]), "q2c": abs(dc["qanet_q2c_drop"])}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8), dpi=150,
                                gridspec_kw={"width_ratios": [1, 1]})

# Panel 1: Absolute ΔF1 (with methodology caveat)
x = np.arange(2); w = 0.32
ax1.bar(x - w/2, [BIDF["c2q"], BIDF["q2c"]], w, label="BiDAF † (retrain abl.)",
        color="#90caf9", edgecolor="white", linewidth=0.8)
ax1.bar(x + w/2, [QANET["c2q"], QANET["q2c"]], w, label="QANet (eval-time abl.)",
        color="#1565c0", edgecolor="white", linewidth=0.8)
for i, (d, q) in enumerate([(BIDF["c2q"], QANET["c2q"]), (BIDF["q2c"], QANET["q2c"])]):
    ax1.text(i - w/2, d + 1.0, f"{d:.1f}", ha="center", fontsize=9, fontweight="bold", color="#42a5f5")
    ax1.text(i + w/2, q + 1.0, f"{q:.1f}", ha="center", fontsize=9, fontweight="bold", color="#0d47a1")
ax1.set_xticks(x); ax1.set_xticklabels(["C2Q\n(C→Q direction)", "Q2C\n(Q→C direction)"],
                                         fontsize=11, fontweight="bold")
ax1.set_ylabel("|ΔF1| upon removal", fontsize=12, fontweight="bold")
ax1.set_title("Direction-Level |ΔF1|", fontsize=13, fontweight="bold")
ax1.legend(fontsize=9, loc="upper right")
ax1.grid(axis="y", alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# Panel 2: Ratio comparison
bidf_ratio = BIDF["c2q"] / max(BIDF["q2c"], 0.01)
qa_ratio = QANET["c2q"] / max(QANET["q2c"], 0.01)
bars = ax2.bar(["BiDAF\n(LSTM)", "QANet\n(Conv+Attn)"], [bidf_ratio, qa_ratio],
               color=["#90caf9", "#1565c0"], edgecolor="white", linewidth=0.8, width=0.5)
ax2.set_ylim(0, qa_ratio * 1.25)
ax2.text(0, bidf_ratio + 1.5, f"{bidf_ratio:.1f}×", ha="center", fontsize=14, fontweight="bold", color="#42a5f5")
ax2.text(1, qa_ratio + 1.5, f"{qa_ratio:.0f}×", ha="center", fontsize=14, fontweight="bold", color="#0d47a1")
ax2.set_ylabel("C2Q / Q2C Ratio", fontsize=12, fontweight="bold")
ax2.set_title("Intra-Architecture Asymmetry Ratio", fontsize=13, fontweight="bold")
ax2.tick_params(axis="x", labelsize=11)
ax2.grid(axis="y", alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.suptitle("H2: Directional Asymmetry — Cross-Architecture Validation", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
os.makedirs("experiments/prism-uploads", exist_ok=True)
fig.savefig("experiments/prism-uploads/h2_fig1_cross_arch.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: experiments/prism-uploads/h2_fig1_cross_arch.png")

# ── Fig 2: Information Density ────────────────────────────────────
sel = _ld2("h2_selective_corruption.json")
assert sel is not None, "h2_selective_corruption.json not found — run H2 Exp 2 first"
assert "ans_only" in sel and "non_ans_only" in sel and "full_context" in sel, \
    f"Expected keys ans_only/non_ans_only/full_context, got: {list(sel.keys())}"

ANS_TE  = sel["ans_only"]["te_mean"]
ANS_TOK = sel["ans_only"]["avg_tokens_corrupted"]
NON_TE  = sel["non_ans_only"]["te_mean"]
NON_TOK = sel["non_ans_only"]["avg_tokens_corrupted"]
FULL_TE = sel["full_context"]["te_mean"]
ANS_DENSITY = ANS_TE / max(ANS_TOK, 1e-12)
NON_DENSITY = NON_TE / max(NON_TOK, 1e-12)
density_ratio = ANS_DENSITY / max(NON_DENSITY, 1e-12)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

# Panel 1: TE by region
cats = ["Answer\nspan", "Non-answer\ntokens", "Full\ncontext"]
tes = [ANS_TE, NON_TE, FULL_TE]
colors = [_PAL["ans"], _PAL["non"], "#616161"]
bars = ax1.bar(cats, tes, color=colors, edgecolor="white", linewidth=0.8, width=0.55)
for b, te, nt in zip(bars, tes, [ANS_TOK, NON_TOK, ANS_TOK+NON_TOK]):
    ax1.text(b.get_x() + b.get_width()/2, te + 0.012,
             f"TE={te:.3f}\n({nt:.0f} tokens)", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax1.set_ylim(0, max(tes) * 1.25)
ax1.set_ylabel("Total Effect (TE)", fontsize=12, fontweight="bold")
ax1.set_title("Corruption Effect by Region", fontsize=13, fontweight="bold")
ax1.grid(axis="y", alpha=0.25); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# Panel 2: Density ratio
bars2 = ax2.bar(["Answer\ntokens", "Non-answer\ntokens"],
                [ANS_DENSITY*1000, NON_DENSITY*1000],
                color=[_PAL["ans"], _PAL["non"]], edgecolor="white", width=0.45)
ax2.text(0, ANS_DENSITY*1000 + 1.5, f"{ANS_DENSITY*1000:.1f}×10⁻³",
         ha="center", fontsize=11, fontweight="bold", color=_PAL["ans"])
ax2.text(1, NON_DENSITY*1000 + 1.5, f"{NON_DENSITY*1000:.2f}×10⁻³",
         ha="center", fontsize=11, fontweight="bold", color=_PAL["non"])
ax2.set_ylim(0, ANS_DENSITY*1000 * 1.25)
ax2.set_ylabel("TE / token  (×10⁻³)", fontsize=12, fontweight="bold")
ax2.set_title("Information Density: Answer ≫ Non-Answer", fontsize=13, fontweight="bold")
ax2.grid(axis="y", alpha=0.25); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

fig.suptitle("H2: Why C2Q Is Irreplaceable — Information Is Spatially Concentrated",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig("experiments/prism-uploads/h2_fig2_info_density.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: experiments/prism-uploads/h2_fig2_info_density.png")

### H3: Necessity of Pointer Asymmetric Wiring — Testing Functional Specialisation

**Research question**: Does the functional specialisation hypothesis implied by the Pointer's asymmetric wiring (M1 shared anchor, M2 for start, M3 for end) hold in the trained model?

**Experiments**: (A) Wiring replacement F1, (B) CKA + cosine similarity, (C) Linear probes for is_start/is_end, (D) Pointer weight quadrant norms, (E) Representation structure analysis

In [ ]:
## H3 Phase A — Pointer Replacement Experiments
from experiments.run_H3 import POINTER_CONFIGS, pointer_forward, run_phase_a, run_phase_b
from experiments.run_H3 import cosine_sim_per_token, linear_cka
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate

print("=" * 70)
print("H3 PHASE A: POINTER WIRING REPLACEMENT EXPERIMENTS")
print("=" * 70)
print("Original design: P(start) = softmax(w1·[M1;M2]), P(end) = softmax(w2·[M1;M3])")
print("Configs tested:")
print("  original  — M1+M2→start, M1+M3→end  (paper design)")
print("  swap      — M1+M3→start, M1+M2→end  (swap M2/M3)")
print("  sym_M2    — M1+M2→start, M1+M2→end  (use M2 for both)")
print("  sym_M3    — M1+M3→start, M1+M3→end  (use M3 for both)")
print("  only_M1   — M1→start, M1→end         (remove M2/M3)")
print("  no_M1     — M2→start, M3→end         (remove M1)")
print(f"\nEval on full dev set ({len(dataset)} samples), batch_size=32.\n")

phase_a = run_phase_a(model, dataset, eval_file, batch_size=32)

os.makedirs("experiments/results/H3", exist_ok=True)

orig_f1 = phase_a["original"]["f1"]
orig_em = phase_a["original"]["em"]

print(f"\n{'='*70}")
print("PHASE A RESULTS")
print(f"{'='*70}")
print(f"{'Config':>12s}  {'F1':>7s}  {'EM':>7s}  {'ΔF1':>8s}  {'ΔEM':>8s}  {'%F1 lost':>9s}")
print("-" * 65)
for cfg, res in sorted(phase_a.items(), key=lambda x: -x[1]["f1"]):
    df1 = res["f1"] - orig_f1
    dem = res["em"] - orig_em
    pct = abs(df1) / orig_f1 * 100 if df1 != 0 else 0
    marker = " ◀ baseline" if cfg == "original" else ""
    print(f"{cfg:>12s}  {res['f1']:7.2f}  {res['em']:7.2f}  {df1:+8.2f}  {dem:+8.2f}  {pct:8.1f}%{marker}")

swap_drop = orig_f1 - phase_a.get("swap", {}).get("f1", orig_f1)
sym2_drop = orig_f1 - phase_a.get("sym_M2", {}).get("f1", orig_f1)
sym3_drop = orig_f1 - phase_a.get("sym_M3", {}).get("f1", orig_f1)
only_m1_drop = orig_f1 - phase_a.get("only_M1", {}).get("f1", orig_f1)
no_m1_drop = orig_f1 - phase_a.get("no_M1", {}).get("f1", orig_f1)
print(f"  5. Asymmetric gain = swap_drop ({swap_drop:.2f}) vs best symmetric ({min(sym2_drop, sym3_drop):.2f})"
      f" → asymmetric wiring {'justified' if swap_drop > 0 else 'not clearly better'}")

In [ ]:
## H3 Phase B — Representation Similarity Analysis
import importlib, experiments.run_H3
importlib.reload(experiments.run_H3)
from experiments.run_H3 import run_phase_b

print("=" * 70)
print("H3 PHASE B: REPRESENTATION SIMILARITY ANALYSIS (M1/M2/M3)")
print("=" * 70)
print("Metrics: Cosine Similarity (per-token, then averaged) + CKA (dataset-level)")
print("CKA = Centered Kernel Alignment — measures representational similarity")
print("  CKA=1 → identical representations, CKA=0 → orthogonal representations\n")

phase_b = run_phase_b(model, dataset, num_samples=1000, batch_size=32, seed=42)

with open("experiments/results/H3/h3_results.json", "w") as f:
    json.dump({"phase_a": phase_a, "phase_b": phase_b}, f, indent=2)

print(f"\n{'='*70}")
print("PHASE B RESULTS")
print(f"{'='*70}")

print(f"\n--- Global Cosine Similarity (all tokens averaged) ---")
print("-" * 55)
for pair, stats in phase_b["global_cosine"].items():
    sim = stats["mean"]
    interp = "very high" if sim > 0.99 else ("high" if sim > 0.95 else ("moderate" if sim > 0.8 else "low"))
    print(f"{pair:>10s}  {sim:8.4f}  {stats['ci95']:8.4f}  ({interp} similarity)")

gc = phase_b["global_cosine"]
m12 = gc.get("M1_M2", {}).get("mean", 0)
m13 = gc.get("M1_M3", {}).get("mean", 0)
m23 = gc.get("M2_M3", {}).get("mean", 0)
print(f"\nOrdering: M2_M3 ({m23:.4f}) > M1_M2 ({m12:.4f}) > M1_M3 ({m13:.4f})")

print(f"\n--- M2 vs M3 Cosine Similarity by Answer Position ---")
print(f"{'Region':>20s}  {'Mean':>8s}  {'±95%CI':>8s}")
print("-" * 45)
pos_data = phase_b.get("position_cosine_M2_M3", {})
for region in ["answer_start", "answer_interior", "answer_end", "non_answer"]:
    s = pos_data.get(region, {})
    print(f"{region:>20s}  {s.get('mean',0):8.4f}  {s.get('ci95',0):8.4f}")

ans_start_sim = pos_data.get("answer_start", {}).get("mean", 0)
ans_end_sim = pos_data.get("answer_end", {}).get("mean", 0)
non_ans_sim = pos_data.get("non_answer", {}).get("mean", 0)
print(f"\nAnswer boundary vs non-answer difference:")
print(f"  start:      {ans_start_sim:.4f} - {non_ans_sim:.4f} = {ans_start_sim - non_ans_sim:+.4f}")
print(f"  end:        {ans_end_sim:.4f} - {non_ans_sim:.4f} = {ans_end_sim - non_ans_sim:+.4f}")
if ans_start_sim < non_ans_sim or ans_end_sim < non_ans_sim:
else:

print(f"\n--- CKA (Centered Kernel Alignment) ---")
cka = phase_b.get("cka", {})
print("-" * 45)
for pair in ["M1_M2", "M1_M3", "M2_M3"]:
    v = cka.get(pair, 0)
    print(f"{pair:>10s}  {v:8.4f}  ({interp})")

cka_12 = cka.get("M1_M2", 0)
cka_13 = cka.get("M1_M3", 0)
cka_23 = cka.get("M2_M3", 0)
print(f"\nCKA ordering: M2_M3 ({cka_23:.4f}) > M1_M2 ({cka_12:.4f}) > M1_M3 ({cka_13:.4f})")
print(f"  M1→M2 change: 1−CKA = {1-cka_12:.4f}")
print(f"  M2→M3 change: 1−CKA = {1-cka_23:.4f}")
print(f"  M1→M3 change: 1−CKA = {1-cka_13:.4f}")

print(f"  1. M2 & M3 share high but not identical representations (CKA={cka_23:.3f} < 1)")
print(f"  3. The asymmetric Pointer design exploits the M2/M3 divergence")

In [ ]:
## H3 — Experiment C: Linear Probe for Start/End Token Specialization
##
## Core claim to verify: "M2 specializes in start-boundary, M3 in end-boundary"
## Phase A (wiring ablation) shows SWAP hurts performance, and Phase B (CKA/cosine)
## shows M2/M3 are different — but neither proves M2 encodes START and M3 encodes END.
##
## This experiment trains linear probes directly:
##   M2 tokens → predict is_start_token (ROC-AUC)
##   M2 tokens → predict is_end_token
##   M3 tokens → predict is_start_token
##   M3 tokens → predict is_end_token
## If AUC(M2→start) > AUC(M3→start) and AUC(M3→end) > AUC(M2→end) → direct evidence

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from tqdm.auto import tqdm

N_PROBE_H3 = 500
random.seed(42); torch.manual_seed(42)
loader_h3_probe = make_loader(dataset, batch_size=1, shuffle=True)

print("=" * 70)
print("H3 EXPERIMENT C: LINEAR PROBE — START/END SPECIALIZATION")
print("=" * 70)
print("Methodology:")
print("  1. Collect M1/M2/M3 token representations for each sample")
print("  2. Labels: is_start (token == answer_start), is_end (token == answer_end)")
print("  3. Train LogisticRegression probes (balanced class weight)")
print("  4. Compare ROC-AUC across M1/M2/M3 for each label type")
print(f"  Samples: {N_PROBE_H3}, 80/20 train/test split\n")

probe_feats = {"M1": {"X": [], "y_start": [], "y_end": []},
               "M2": {"X": [], "y_start": [], "y_end": []},
               "M3": {"X": [], "y_start": [], "y_end": []}}

samples_h3p = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(
            loader_h3_probe, total=N_PROBE_H3, desc="H3 Probe features"):
        if samples_h3p >= N_PROBE_H3:
            break
        y1_val, y2_val = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1_val >= L or y2_val >= L or y1_val > y2_val:
            continue

        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, _, intermediates = qanet_forward(
            model, Cwid, Ccid, Qwid, Qcid, collect=False)
        M1, M2, M3 = intermediates["M1"], intermediates["M2"], intermediates["M3"]

        pad_mask = (Cwid[0].cpu() != 0).numpy()
        label_start = np.zeros(L, dtype=int)
        label_end = np.zeros(L, dtype=int)
        label_start[y1_val] = 1
        label_end[y2_val] = 1

        for name, M in [("M1", M1), ("M2", M2), ("M3", M3)]:
            feat = M[0].cpu().float().numpy()  # [d_model, L]
            for t in range(L):
                if not pad_mask[t]:
                    continue
                probe_feats[name]["X"].append(feat[:, t])
                probe_feats[name]["y_start"].append(label_start[t])
                probe_feats[name]["y_end"].append(label_end[t])

        samples_h3p += 1

print(f"\nCollected {samples_h3p} samples")
for name in ["M1", "M2", "M3"]:
    n_total = len(probe_feats[name]["X"])
    n_start = sum(probe_feats[name]["y_start"])
    n_end = sum(probe_feats[name]["y_end"])
    print(f"  {name}: {n_total} tokens, {n_start} start+ ({n_start/n_total*100:.2f}%), "
          f"{n_end} end+ ({n_end/n_total*100:.2f}%)")

print(f"\n{'='*70}")
print("LINEAR PROBE RESULTS — START/END TOKEN PREDICTION")
print(f"{'='*70}")

probe_results_h3 = {}
print(f"\n{'Repr':>4s}  {'Target':>8s}  {'AUC':>6s}  {'Acc':>6s}  {'F1(+)':>7s}  {'N_test':>7s}")
print("-" * 50)

for name in ["M1", "M2", "M3"]:
    X_all = np.array(probe_feats[name]["X"])
    y_start_all = np.array(probe_feats[name]["y_start"])
    y_end_all = np.array(probe_feats[name]["y_end"])

    n = len(X_all)
    idx = np.arange(n)
    np.random.seed(42)
    np.random.shuffle(idx)
    split = int(0.8 * n)

    for target_name, y_all in [("start", y_start_all), ("end", y_end_all)]:
        X_tr, X_te = X_all[idx[:split]], X_all[idx[split:]]
        y_tr, y_te = y_all[idx[:split]], y_all[idx[split:]]

        if y_tr.sum() < 5 or y_te.sum() < 5:
            print(f"{name:>4s}  {target_name:>8s}  (insufficient positives)")
            continue

        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, solver="lbfgs")
        clf.fit(X_tr, y_tr)
        y_prob = clf.predict_proba(X_te)[:, 1]
        y_pred = clf.predict(X_te)

        auc = roc_auc_score(y_te, y_prob)
        acc = accuracy_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred)

        probe_results_h3[(name, target_name)] = {"auc": auc, "acc": acc, "f1": f1}
        print(f"{name:>4s}  {target_name:>8s}  {auc:6.3f}  {acc:6.3f}  {f1:7.3f}  {len(y_te):7d}")
    print()

print(f"--- Specialization Analysis ---")
auc_m2_start = probe_results_h3.get(("M2", "start"), {}).get("auc", 0)
auc_m3_start = probe_results_h3.get(("M3", "start"), {}).get("auc", 0)
auc_m2_end = probe_results_h3.get(("M2", "end"), {}).get("auc", 0)
auc_m3_end = probe_results_h3.get(("M3", "end"), {}).get("auc", 0)
auc_m1_start = probe_results_h3.get(("M1", "start"), {}).get("auc", 0)
auc_m1_end = probe_results_h3.get(("M1", "end"), {}).get("auc", 0)

print(f"\nStart token prediction:")
print(f"  M1: AUC={auc_m1_start:.3f}  M2: AUC={auc_m2_start:.3f}  M3: AUC={auc_m3_start:.3f}")
      f"(Δ = {abs(auc_m2_start - auc_m3_start):.3f})")

print(f"\nEnd token prediction:")
print(f"  M1: AUC={auc_m1_end:.3f}  M2: AUC={auc_m2_end:.3f}  M3: AUC={auc_m3_end:.3f}")
      f"(Δ = {abs(auc_m3_end - auc_m2_end):.3f})")

m2_start_adv = auc_m2_start - auc_m3_start
m3_end_adv = auc_m3_end - auc_m2_end

print(f"\nSpecialization pattern:")
print(f"  M2 start advantage: {m2_start_adv:+.3f} AUC")
print(f"  M3 end advantage:   {m3_end_adv:+.3f} AUC")

if m2_start_adv > 0.01 and m3_end_adv > 0.01:
elif m2_start_adv > 0 and m3_end_adv > 0:
else:

print(f"  M1→M2→M3 start AUC: {auc_m1_start:.3f} → {auc_m2_start:.3f} → {auc_m3_start:.3f}")
print(f"  M1→M2→M3 end AUC:   {auc_m1_end:.3f} → {auc_m2_end:.3f} → {auc_m3_end:.3f}")
if auc_m3_start > auc_m1_start and auc_m3_end > auc_m1_end:

print(f"\n--- Limitations ---")
print(f"  1. Linear probes only detect linearly decodable information")
print(f"  2. Extreme class imbalance (~{samples_h3p}/{len(probe_feats['M1']['X']):.0f} ≈ "
      f"{samples_h3p/len(probe_feats['M1']['X'])*100:.2f}% positive)")
print(f"  3. Specialization may exist in non-linear subspaces not captured here")
print(f"  4. The Pointer layer uses bilinear projection, not linear readout")

os.makedirs("experiments/results/H3", exist_ok=True)
h3_probe_save = {f"{k[0]}_{k[1]}": v for k, v in probe_results_h3.items()}
h3_probe_save["specialization"] = {
    "m2_start_advantage": float(m2_start_adv),
    "m3_end_advantage": float(m3_end_adv),
}
with open("experiments/results/H3/h3_linear_probe.json", "w") as f:
    json.dump(h3_probe_save, f, indent=2)
print(f"\nSaved to experiments/results/H3/h3_linear_probe.json")

### Stage 3 — Results Summary

All experiment results are saved to `experiments/results/`.

## H3 — Experiment D: Pointer Weight Quadrant Analysis

w1 = [2d] where first d dims → M1, last d dims → M2.
w2 = [2d] where first d dims → M1, last d dims → M3.
Compare L2 norms to see if the model learned to downweight M1.

In [ ]:
## H3 Exp D — Pointer Weight Quadrant Analysis
## w1 shape: [2*d_model] → first d = M1 quadrant, last d = M2 quadrant
## w2 shape: [2*d_model] → first d = M1 quadrant, last d = M3 quadrant

print("=" * 70)
print("H3 EXPERIMENT D: POINTER WEIGHT QUADRANT ANALYSIS")
print("=" * 70)

w1 = model.out.w1.detach().cpu().float()
w2 = model.out.w2.detach().cpu().float()
d = w1.shape[0] // 2

w1_m1 = w1[:d]
w1_m2 = w1[d:]
w2_m1 = w2[:d]
w2_m3 = w2[d:]

w1_m1_norm = w1_m1.norm(2).item()
w1_m2_norm = w1_m2.norm(2).item()
w2_m1_norm = w2_m1.norm(2).item()
w2_m3_norm = w2_m3.norm(2).item()

print(f"\nd_model = {d}")
print(f"\n{'Weight':<8} {'M1 quadrant L2':>16} {'M2/M3 quadrant L2':>20} {'Ratio (M2M3/M1)':>18}")
print("-" * 66)
print(f"{'w1':<8} {w1_m1_norm:>16.4f} {w1_m2_norm:>20.4f} {w1_m2_norm/w1_m1_norm:>18.2f}x")
print(f"{'w2':<8} {w2_m1_norm:>16.4f} {w2_m3_norm:>20.4f} {w2_m3_norm/w2_m1_norm:>18.2f}x")

print(f"\n--- Interpretation ---")
if w1_m2_norm > w1_m1_norm and w2_m3_norm > w2_m1_norm:
    print(f"  Both w1 and w2 assign HIGHER norm to M2/M3 quadrants than M1 quadrants")
elif w1_m1_norm > w1_m2_norm or w2_m1_norm > w2_m3_norm:
    print(f"  M1 quadrant has comparable or higher norm in at least one weight")
else:
    print(f"  Mixed pattern — see values above")

# Also check: absolute magnitude of M1 quadrant entries
w1_m1_mean_abs = w1_m1.abs().mean().item()
w1_m2_mean_abs = w1_m2.abs().mean().item()
w2_m1_mean_abs = w2_m1.abs().mean().item()
w2_m3_mean_abs = w2_m3.abs().mean().item()

print(f"\n--- Mean |weight| per dimension ---")
print(f"  w1: M1={w1_m1_mean_abs:.6f}, M2={w1_m2_mean_abs:.6f} (ratio={w1_m2_mean_abs/w1_m1_mean_abs:.2f}x)")
print(f"  w2: M1={w2_m1_mean_abs:.6f}, M3={w2_m3_mean_abs:.6f} (ratio={w2_m3_mean_abs/w2_m1_mean_abs:.2f}x)")

h3_weight_results = {
    "d_model": d,
    "w1_M1_L2": w1_m1_norm, "w1_M2_L2": w1_m2_norm, "w1_ratio": w1_m2_norm / w1_m1_norm,
    "w2_M1_L2": w2_m1_norm, "w2_M3_L2": w2_m3_norm, "w2_ratio": w2_m3_norm / w2_m1_norm,
    "w1_M1_mean_abs": w1_m1_mean_abs, "w1_M2_mean_abs": w1_m2_mean_abs,
    "w2_M1_mean_abs": w2_m1_mean_abs, "w2_M3_mean_abs": w2_m3_mean_abs,
}
os.makedirs("experiments/results/H3", exist_ok=True)
with open("experiments/results/H3/h3_weight_quadrants.json", "w") as f:
    json.dump(h3_weight_results, f, indent=2)
print(f"\nSaved to experiments/results/H3/h3_weight_quadrants.json")

## H3 — Experiment E: Representation Structure Analysis (M1/M2/M3)

Measures local contrast, answer boundary sharpness, and token norm CoV
across M1/M2/M3 to characterize what the third iteration does to representations.

In [ ]:
## H3 Exp E — Representation Structure Analysis
## Metrics computed on M1/M2/M3 (200 samples):
##   1. Local contrast: 1 - cosine_sim(token_t, token_{t+1}), averaged
##   2. Answer boundary sharpness: cosine distance at y1/y2 boundaries
##   3. Token norm CoV: std(norms) / mean(norms)

import importlib, experiments.tracer
importlib.reload(experiments.tracer)
from experiments.tracer import qanet_forward
from tqdm.auto import tqdm
import torch.nn.functional as F

N_STRUCT = 200

print("=" * 70)
print("H3 EXPERIMENT E: REPRESENTATION STRUCTURE ANALYSIS")
print("=" * 70)
print(f"Samples: {N_STRUCT}")
print("Metrics: local contrast, answer boundary sharpness, token norm CoV\n")

random.seed(42); torch.manual_seed(42)
loader_struct = make_loader(dataset, batch_size=1, shuffle=True)

metrics = {name: {"local_contrast": [], "boundary_sharpness": [], "norm_cov": []}
           for name in ["M1", "M2", "M3"]}

n_collected = 0
with torch.no_grad():
    for Cwid, Ccid, Qwid, Qcid, y1, y2, ids in tqdm(
            loader_struct, total=N_STRUCT, desc="H3 Exp E: Repr Structure"):
        if n_collected >= N_STRUCT:
            break
        y1v, y2v = y1.item(), y2.item()
        L = Cwid.size(1)
        if y1v >= L or y2v >= L or y1v > y2v or y1v < 1 or y2v >= L - 1:
            continue

        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)

        _, _, _, intermediates = qanet_forward(
            model, Cwid, Ccid, Qwid, Qcid, collect=False)

        pad_mask = (Cwid[0].cpu() != 0)
        valid_len = pad_mask.sum().item()

        for name in ["M1", "M2", "M3"]:
            M = intermediates[name][0].cpu().float()  # [d_model, L]
            M_valid = M[:, :valid_len]  # [d, valid_len]
            M_t = M_valid.T  # [valid_len, d]

            # 1. Local contrast: 1 - cosine_sim(t, t+1)
            if valid_len > 1:
                norms = M_t.norm(dim=1, keepdim=True).clamp(min=1e-8)
                M_normed = M_t / norms
                cos_adj = (M_normed[:-1] * M_normed[1:]).sum(dim=1)  # [valid_len-1]
                local_contrast = (1 - cos_adj).mean().item()
            else:
                local_contrast = 0.0

            # 2. Answer boundary sharpness
            #    cosine distance between boundary token and its neighbor OUTSIDE the span
            norms_flat = M_t.norm(dim=1).clamp(min=1e-8)
            def cos_dist(i, j):
                return 1 - (M_t[i] @ M_t[j]) / (norms_flat[i] * norms_flat[j])

            sharp_start = cos_dist(y1v, y1v - 1).item()  # start vs token before
            sharp_end = cos_dist(y2v, y2v + 1).item()    # end vs token after
            boundary_sharpness = (sharp_start + sharp_end) / 2

            # 3. Token norm CoV: std / mean of L2 norms
            token_norms = M_valid.norm(dim=0)  # [valid_len]
            norm_cov = (token_norms.std() / token_norms.mean()).item()

            metrics[name]["local_contrast"].append(local_contrast)
            metrics[name]["boundary_sharpness"].append(boundary_sharpness)
            metrics[name]["norm_cov"].append(norm_cov)

        n_collected += 1

print(f"\nCollected {n_collected} samples")
print(f"\n{'='*70}")
print("REPRESENTATION STRUCTURE RESULTS")
print(f"{'='*70}")
print(f"\n{'Metric':<25} {'M1':>10} {'M2':>10} {'M3':>10} {'M2→M3 Δ':>12} {'M2→M3 %':>10}")
print("-" * 80)

struct_results = {}
for metric_name in ["local_contrast", "boundary_sharpness", "norm_cov"]:
    vals = {}
    for name in ["M1", "M2", "M3"]:
        arr = np.array(metrics[name][metric_name])
        vals[name] = {"mean": arr.mean(), "ci95": 1.96 * arr.std() / np.sqrt(len(arr))}
    
    m2_val = vals["M2"]["mean"]
    m3_val = vals["M3"]["mean"]
    delta = m3_val - m2_val
    pct = (delta / m2_val * 100) if m2_val != 0 else 0
    
    print(f"{metric_name:<25} {vals['M1']['mean']:>10.6f} {vals['M2']['mean']:>10.6f} "
          f"{vals['M3']['mean']:>10.6f} {delta:>+12.6f} {pct:>+9.2f}%")
    
    struct_results[metric_name] = {
        n: {"mean": float(vals[n]["mean"]), "ci95": float(vals[n]["ci95"])} 
        for n in ["M1", "M2", "M3"]
    }
    struct_results[metric_name]["M2_to_M3_delta"] = float(delta)
    struct_results[metric_name]["M2_to_M3_pct"] = float(pct)

print(f"\n--- Interpretation ---")
lc_delta = struct_results["local_contrast"]["M2_to_M3_delta"]
bs_delta = struct_results["boundary_sharpness"]["M2_to_M3_delta"]
nc_delta = struct_results["norm_cov"]["M2_to_M3_delta"]

signs = [lc_delta < 0, bs_delta < 0, nc_delta < 0]
n_smoothing = sum(signs)

elif n_smoothing <= 1:

# Magnitude check
lc_pct = abs(struct_results["local_contrast"]["M2_to_M3_pct"])
bs_pct = abs(struct_results["boundary_sharpness"]["M2_to_M3_pct"])
nc_pct = abs(struct_results["norm_cov"]["M2_to_M3_pct"])
avg_pct = (lc_pct + bs_pct + nc_pct) / 3

print(f"\n  Average |M2→M3 change|: {avg_pct:.2f}%")

os.makedirs("experiments/results/H3", exist_ok=True)
with open("experiments/results/H3/h3_repr_structure.json", "w") as f:
    json.dump(struct_results, f, indent=2)
print(f"\nSaved to experiments/results/H3/h3_repr_structure.json")

In [ ]:
## H3 — Publication Figures
## Fig 1: Wiring Replacement ΔF1 (tests the three design assumptions)
## Fig 2: Representation Structure — M1→M2→M3 progressive smoothing
## Requires: h3_results.json, h3_repr_structure.json

import json, os
import numpy as np
import matplotlib.pyplot as plt

_H3_DIR = "experiments/results/H3"
_PAL = {"main": "#1565c0", "neg": "#c62828", "neutral": "#616161",
        "m1": "#9e9e9e", "m2": "#1976d2", "m3": "#d32f2f", "accent": "#7b1fa2"}

def _ld3(name):
    p = os.path.join(_H3_DIR, name)
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f)

# ── Fig 1: Wiring Replacement — F1 scores ────────────────────────
DISPLAY = {
    "original": "original\n(baseline)",
    "sym_M2":   "sym_M2\n(M2 both)",
    "sym_M3":   "sym_M3\n(M3 both)",
    "no_M1":    "no_M1\n(drop M1)",
    "swap":     "swap\n(M2↔M3)",
    "only_M1":  "only_M1\n(no M2/M3)",
}
order = ["original", "sym_M2", "sym_M3", "no_M1", "swap", "only_M1"]

h3r = _ld3("h3_results.json")
assert h3r and "phase_a" in h3r, "h3_results.json not found — run Phase A first"
pa = h3r["phase_a"]

baseline = pa["original"]["f1"]
f1s = [pa[c]["f1"] for c in order]
labels = [DISPLAY[c] for c in order]
colors = ["#1565c0" if c == "original" else ("#90CAF9" if pa[c]["f1"] >= 69.0 else "#c62828") for c in order]

fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
bars = ax.bar(range(len(order)), f1s, color=colors, edgecolor="white", linewidth=0.8, width=0.6)

for i, (c, f1) in enumerate(zip(order, f1s)):
    delta = f1 - baseline
    lbl = f"{f1:.2f}" if delta == 0 else f"{f1:.2f}  ({delta:+.2f})"
    ax.text(i, f1 + 0.25, lbl, ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax.axhline(baseline, color="#1565c0", lw=1.0, ls="--", alpha=0.4)
ax.set_ylim(62, 72)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(labels, fontsize=9, fontweight="bold")
ax.set_ylabel("F1 Score", fontsize=12, fontweight="bold")
ax.set_title("Pointer Wiring Replacement: Asymmetric Design Provides ≤0.23 F1 Advantage",
             fontsize=13, fontweight="bold")

ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
os.makedirs("experiments/prism-uploads", exist_ok=True)
fig.savefig("experiments/prism-uploads/h3_fig1_wiring_replacement.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: experiments/prism-uploads/h3_fig1_wiring_replacement.png")

# ── Fig 2: Representation Structure — Progressive Smoothing ──────
rs = _ld3("h3_repr_structure.json")
assert rs is not None, "h3_repr_structure.json not found — run H3 Exp E first"

metric_map = {"Local Contrast": "local_contrast", "Boundary Sharpness": "boundary_sharpness", "Norm CoV": "norm_cov"}
STRUCT = {}
for display, key in metric_map.items():
    assert key in rs, f"{key} missing in h3_repr_structure.json"
    STRUCT[display] = {}
    for mk in ["M1", "M2", "M3"]:
        v = rs[key][mk]
        STRUCT[display][mk] = v["mean"] if isinstance(v, dict) else v

metrics = list(STRUCT.keys())
m1_vals = [STRUCT[m]["M1"] for m in metrics]
m2_vals = [STRUCT[m]["M2"] for m in metrics]
m3_vals = [STRUCT[m]["M3"] for m in metrics]

x = np.arange(len(metrics))
w = 0.25

fig, ax = plt.subplots(figsize=(8.5, 5), dpi=150)
ax.bar(x - w, m1_vals, w, label="M₁  (7 layers)", color=_PAL["m1"], edgecolor="white", linewidth=0.8)
ax.bar(x,     m2_vals, w, label="M₂  (14 layers)", color=_PAL["m2"], edgecolor="white", linewidth=0.8)
ax.bar(x + w, m3_vals, w, label="M₃  (21 layers)", color=_PAL["m3"], edgecolor="white", linewidth=0.8)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11, fontweight="bold")
ax.set_ylabel("Metric Value (higher = more discriminable)", fontsize=11, fontweight="bold")
ax.set_title("Third Iteration Smooths Without Adding Information",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10, loc="upper right", framealpha=0.92)
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig("experiments/prism-uploads/h3_fig2_repr_structure.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: experiments/prism-uploads/h3_fig2_repr_structure.png")